### 1. Setup and configuration
All knobs are in `CFG`. `P7_CFG_OVERRIDE` (JSON in an env var) overrides them without editing the cell.

In [ ]:

# Setup and configuration

import os, sys, json, re, math, time, csv, random, zipfile, shutil, hashlib, warnings, gc, types, subprocess
from dataclasses import dataclass, asdict, fields
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np

def _ensure(pip_name, import_name):
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

_ensure("tokenizers", "tokenizers")
_ensure("huggingface_hub", "huggingface_hub")
_ensure("scikit-learn", "sklearn")
_ensure("scipy", "scipy")
_ensure("pandas", "pandas")

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint

warnings.filterwarnings("ignore", category=UserWarning)

# Configuration
# Optional quick test: set P7_CFG_OVERRIDE before running.

CFG = dict(
    # identity
    EXP_NAME='m3_lormaplus_maxsim',
    P7_TAG='p7_m3_lormaplus_maxsim_v1',
    P1_TAG='decoder_only_rope_gqa_swiglu_mtp4_v1',
    VARIANT='lormaplus',
    # where things are (None = auto-discover under /kaggle/input)
    P1_CKPT=None,
    P1_TOKENIZER_DIR=None,
    TOK_KIND='auto',
    TOK_PROBE_DOCS=24,
    TOK_PROBE_STRICT=True,
    P7_ROOT=None,
    HF_DATASET_REPO='Exploration-Lab/CS-781-Capstone',
    # long-document handling
    STRIP_PLACEHOLDERS=True,
    MAX_CHARS=100000,
    CHUNK_LEN=512,
    CHUNK_STRIDE=512,
    MIN_TAIL=64,
    MAX_CHUNKS=32,
    ROPE_PI=False,
    # document representation
    TOKEN_POOL='mean',
    DOC_MODE='pooled',
    DOC_POOL='mean',
    PROJ_DIM=None,
    MAXSIM=True,
    W_POOLED=1.0,
    W_MAXSIM=1.0,
    # adaptation technique
    LORMA_R=32,
    LORMA_ALPHA=64,
    LORMA_DROPOUT=0.05,
    LORMA_TARGETS=('q_proj', 'k_proj', 'v_proj', 'o_proj', 'w_gate', 'w_up', 'w_down'),
    # training
    TRAIN=True,
    EPOCHS=8,
    BATCH_Q=8,
    NPOS=2,
    HARD_NEGS=2,
    HARD_POOL=30,
    DENSE_SKIP=3,
    REFRESH_NEGS_AT=(2, 4, 6),
    TRAIN_CHUNKS=6,
    TEMP=0.05,
    LR=0.0003,
    HEAD_LR=0.0001,
    WEIGHT_DECAY=0.01,
    WARMUP_FRAC=0.1,
    MIN_LR_RATIO=0.1,
    GRAD_CLIP=1.0,
    GRAD_CKPT=True,
    TOK_BUDGET_TRAIN=16384,
    TOK_BUDGET_EVAL=32768,
    EVAL_EVERY=1,
    PATIENCE=3,
    MAX_RUNTIME_H=10.5,
    # stage 0: self-supervised same-case contrast (0 = off)
    SSL_EPOCHS=0,
    SSL_BATCH=32,
    SSL_LR_SCALE=1.0,
    # lexical signals + fusion (weights are tuned on dev)
    USE_BM25=True,
    USE_REFS=True,
    USE_PRF=True,
    BM25_NGRAM=(1, 1),
    BM25_K1=1.2,
    BM25_B=0.75,
    MIN_GAIN=0.002,
    # evaluation / output
    K=7,
    SEED=1234,
    RUN_ZERO_SHOT=True,
    RUN_PREDICT=True,
    MAX_QUERIES=None,
    MAX_POOL=None,
)
_ov = os.environ.get("P7_CFG_OVERRIDE")
if _ov:
    CFG.update(json.loads(_ov))
for _k in ("LORMA_TARGETS", "BM25_NGRAM", "REFRESH_NEGS_AT"):          # JSON gives lists; not every variant has every key
    if _k in CFG:
        CFG[_k] = tuple(CFG[_k])
C = types.SimpleNamespace(**CFG)
assert C.TOKEN_POOL in ("mean", "last") and C.DOC_POOL in ("mean", "max", "first", "attn")
assert C.DOC_MODE in ("pooled", "hier") and C.VARIANT in ("lormaplus", "partial", "full")

# Paths and device

IS_KAGGLE = os.path.isdir("/kaggle/working")
INPUT_ROOT = "/kaggle/input"
BASE_DIR = "/kaggle/working" if IS_KAGGLE else "."
WORK_DIR = os.path.join(BASE_DIR, "p7_" + C.EXP_NAME)
CKPT_DIR = os.path.join(WORK_DIR, "checkpoints")
OUT_DIR = os.path.join(BASE_DIR, "cs781_phase7_" + C.EXP_NAME)       # prediction.json + submission.zip land here
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = {"enabled": False, "dtype": torch.float16}
if DEVICE.type == "cuda":
    AMP["enabled"] = True
    AMP["dtype"] = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16

def amp_ctx():
    return torch.autocast(device_type="cuda", dtype=AMP["dtype"], enabled=AMP["enabled"])

def log(*a):
    print(*a, flush=True)

def fmt_time(sec):
    sec = int(max(sec, 0))
    return f"{sec // 3600}h{(sec % 3600) // 60:02d}m{sec % 60:02d}s"

def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def walk_find(filename, roots):
    """all directories under `roots` that contain a file called `filename`"""
    hits, seen = [], set()
    for root in roots:
        if not root or not os.path.isdir(root):
            continue
        for d, _, files in os.walk(root):
            if filename in files and d not in seen:
                seen.add(d)
                hits.append(d)
    return hits

def read_json(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None

def atomic_json(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=1)
    os.replace(tmp, path)

def atomic_torch_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)

def hf_token():

    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("HF")
        if t:
            return t
    except Exception:
        pass
    return os.environ.get("HF_TOKEN") or os.environ.get("HF") or None

HF_TOKEN = hf_token()
set_seed(C.SEED)
RESULTS = {"config": {k: (list(v) if isinstance(v, tuple) else v) for k, v in CFG.items()}}
T_START = time.time()

log(f"torch {torch.__version__} | device {DEVICE} | "
    f"{torch.cuda.get_device_name() if DEVICE.type == 'cuda' else 'NO GPU (this will be very slow)'} | "
    f"autocast {AMP['dtype'] if AMP['enabled'] else 'off'}")
log(f"experiment {C.EXP_NAME} | variant {C.VARIANT} | HF token available: {bool(HF_TOKEN)} | output dir {OUT_DIR}")

### 2. Phase-1 Model 3
The architecture classes are copied from the Model 3 notebook so the checkpoint loads with `strict=True`; the unused multi-token-prediction heads are dropped afterwards.

In [ ]:


# Differences to Model 2 that matter here:

# * 50,000-id tokenizer, embedding rows padded to 50,048.

from dataclasses import field

@dataclass
class ModelConfig:
    vocab_size: int = 50_048                      # embedding rows (padded)
    real_vocab: int = 50_000                      # ids that the tokenizer can produce
    d_model: int = 640
    n_layers: int = 8
    n_heads: int = 10
    n_kv_heads: int = 2
    ffn_hidden: int = 1536
    dropout: float = 0.1
    max_len: int = 1024 + 64
    mtp_heads: int = 3
    mtp_weights: list = field(default_factory=lambda: [1.0, 0.5, 0.25])
    pad_id: int = 0
    unk_id: int = 1
    eos_id: int = 2
    bos_id: int = 3
    mask_id: int = 4

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):                                           # as in Model 3
        variance = x.float().pow(2).mean(-1, keepdim=True)
        inv_rms = torch.rsqrt(variance + self.eps).to(x.dtype)
        return x * inv_rms * self.weight.to(x.dtype)

def build_rope(head_dim, max_len, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    freqs = torch.outer(torch.arange(max_len).float(), inv_freq)
    return freqs.cos(), freqs.sin()          # each (max_len, head_dim/2)

def rope_tables(head_dim, n_pos, scale=1.0, base=10000.0):
    """RoPE cos/sin tables for positions 0..n_pos-1 divided by `scale` (scale > 1 = linear position interpolation)"""
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    freqs = torch.outer(torch.arange(n_pos).float() / scale, inv_freq)
    return freqs.cos(), freqs.sin()

def extend_rope(lm, n_pos, scale):
    cos, sin = rope_tables(lm.cfg.d_model // lm.cfg.n_heads, n_pos, scale)
    lm.rope_cos = cos.to(lm.embed.weight.device)
    lm.rope_sin = sin.to(lm.embed.weight.device)
    lm.cfg.max_len = n_pos

def apply_rope(x, cos, sin):

    x1, x2 = x.float().chunk(2, dim=-1)
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.type_as(x)

def repeat_kv(x, groups):
    if groups == 1:
        return x
    B, KV, T, hd = x.shape
    return x[:, :, None].expand(B, KV, groups, T, hd).reshape(B, KV * groups, T, hd)

class GQAttention(nn.Module):
    """causal grouped-query self-attention with RoPE (10 query heads share 2 key/value heads)"""

    def __init__(self, cfg):
        super().__init__()
        self.h = cfg.n_heads
        self.kv = cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.groups = self.h // self.kv
        self.q_proj = nn.Linear(cfg.d_model, self.h * self.hd, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.o_proj = nn.Linear(self.h * self.hd, cfg.d_model, bias=False)

    def _qkv(self, x):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        return q, k, v

    def forward(self, x, cos, sin):
        B, T, _ = x.shape
        q, k, v = self._qkv(x)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)
        k, v = repeat_kv(k, self.groups), repeat_kv(v, self.groups)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1))

class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w_gate = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_up = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_down = nn.Linear(cfg.ffn_hidden, cfg.d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn = GQAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.ffn = SwiGLU(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, cos, sin):
        x = x + self.drop(self.attn(self.norm1(x), cos, sin))
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x

class MTPHead(nn.Module):
    """residual projection + RMSNorm:  z = RMSNorm(x + W x);  the logits are z @ E^T with the tied embedding E"""

    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim, dim, bias=False)
        self.norm = RMSNorm(dim)

    def forward(self, x):
        return self.norm(x + self.proj(x))

class DecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)       # also the (tied) output projection
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.heads = nn.ModuleList([MTPHead(cfg.d_model) for _ in range(cfg.mtp_heads)])
        self.drop = nn.Dropout(cfg.dropout)
        cos, sin = build_rope(cfg.d_model // cfg.n_heads, cfg.max_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def _rope(self, T, pos_ids=None):
        if pos_ids is None:
            return self.rope_cos[:T][None, None], self.rope_sin[:T][None, None]
        return self.rope_cos[pos_ids][:, None], self.rope_sin[pos_ids][:, None]

    def final_norm(self, x):
        """the retrieval features = what the ordinary next-token head (MTP head 0) reads"""
        return self.heads[0](x)

def count_params(model, trainable_only=False):
    return sum(p.numel() for p in model.parameters() if (p.requires_grad or not trainable_only))

# Load the Phase-1 checkpoint

def find_phase1_ckpt():
    if C.P1_CKPT:
        assert os.path.isfile(C.P1_CKPT), f"P1_CKPT not found: {C.P1_CKPT}"
        return C.P1_CKPT
    roots = [INPUT_ROOT, os.getcwd()]
    best = None                                                    # (val_loss, path)
    for d in walk_find("best.pt", roots):
        meta = read_json(os.path.join(d, "best_meta.json")) or {}
        if meta.get("tag") != C.P1_TAG:
            continue
        key = meta.get("val_loss", 1e9)
        if best is None or key < best[0]:
            best = (key, os.path.join(d, "best.pt"))
    if best is None:                                               # fall back to a latest.pt of this model
        for d in walk_find("latest.pt", roots):
            meta = read_json(os.path.join(d, "latest_meta.json")) or {}
            if meta.get("tag") != C.P1_TAG:
                continue
            key = -meta.get("global_step", 0)
            if best is None or key < best[0]:
                best = (key, os.path.join(d, "latest.pt"))
    if best is None:
        raise FileNotFoundError(
            "Phase-1 Model-3 checkpoint not found. Attach the Phase-1 Model-3 output (the folder holding best.pt and "
            f"best_meta.json with tag '{C.P1_TAG}') as a Kaggle input, or set P1_CKPT.")
    return best[1]

def load_phase1():
    ck_path = find_phase1_ckpt()
    log(f"Phase-1 checkpoint: {ck_path}")
    ck = torch.load(ck_path, map_location="cpu", weights_only=False)
    names = {f.name for f in fields(ModelConfig)}
    cfg = ModelConfig(**{k: v for k, v in (ck.get("config") or {}).items() if k in names})
    sd = {(k[len("_orig_mod."):] if k.startswith("_orig_mod.") else k): v for k, v in ck["model"].items()}
    cfg.vocab_size = int(sd["embed.weight"].shape[0])              # the weights are the authority on the embedding size
    cfg.mtp_heads = len({k.split(".")[1] for k in sd if k.startswith("heads.")})
    cfg.dropout = 0.0                                              # no dropout in the frozen base
    model = DecoderLM(cfg)
    model.load_state_dict(sd, strict=True)                         # raises on ANY key / shape mismatch
    n_all = count_params(model)
    log(f"loaded Phase-1 weights strictly | tag={ck.get('tag')} | {n_all / 1e6:.2f}M params | d_model={cfg.d_model} "
        f"layers={cfg.n_layers} heads={cfg.n_heads}/{cfg.n_kv_heads} ffn={cfg.ffn_hidden} max_len={cfg.max_len} "
        f"vocab rows={cfg.vocab_size} (real {cfg.real_vocab}) mtp_heads={cfg.mtp_heads}")
    model.heads = nn.ModuleList([model.heads[0]])                  # retrieval needs the trunk + head 0 only
    cfg.mtp_heads = 1
    dropped = n_all - count_params(model)
    log(f"dropped the unused multi-token-prediction heads: -{dropped / 1e6:.2f}M params -> {count_params(model) / 1e6:.2f}M "
        f"parameters are the base model of this notebook (percentages below are relative to this)")
    return model, cfg, ck_path, ck.get("tag"), dropped

P1_MODEL, P1_CFG, P1_CKPT_PATH, P1_CKPT_TAG, P1_DROPPED_PARAMS = load_phase1()
assert C.ROPE_PI or C.CHUNK_LEN <= P1_CFG.max_len, f"CHUNK_LEN {C.CHUNK_LEN} exceeds the Phase-1 max_len {P1_CFG.max_len} (set ROPE_PI=True to extend the context)"
if P1_CKPT_TAG != C.P1_TAG:
    log(f"WARNING: checkpoint tag {P1_CKPT_TAG!r} != expected {C.P1_TAG!r} (weights did load, so the architecture matches)")
RESULTS["phase1"] = {"ckpt": P1_CKPT_PATH, "tag": P1_CKPT_TAG, "params": count_params(P1_MODEL), "unused_mtp_params_dropped": P1_DROPPED_PARAMS}

### 2b. Tokenizer candidates
Every tokenizer under `/kaggle/input` whose vocabulary size equals the model's is a candidate; the next cell lets the model choose.

In [ ]:

# CELL 2b - tokenizer candidates

# tokenizer gives ~8-11 nats, the right one a few).

WORD_PATTERN = re.compile(
    r"""'s|'t|'re|'ve|'m|'ll|'d|[a-zA-Z]+|[0-9]+|[^\s\a-zA-Z0-9]+"""
)

class LegalBPE:
    def __init__(self, vocab_path, merges_path, meta_path=None):
        # Load vocabulary
        with open(vocab_path, "r", encoding="utf-8") as f:
            self.vocab = json.load(f)

        # Load merge rules
        with open(merges_path, "r", encoding="utf-8") as f:
            merges = json.load(f)

         # Tokenizer fingerprint for window-cache validation
        h = hashlib.md5()

        for path in (vocab_path, merges_path, meta_path):
            if path and os.path.exists(path):
                with open(path, "rb") as f:
                    h.update(f.read())

        self.fingerprint = h.hexdigest()[:12]

        # Merge rank: lower rank = earlier BPE merge
        self.ranks = {
            (a, b): i
            for i, (a, b) in enumerate(merges)
        }

        # ID -> token
        self.inv = {
            int(i): t
            for t, i in self.vocab.items()
        }

        self.vocab_size = max(self.inv) + 1

        assert len(self.inv) == self.vocab_size, \
            "vocab ids are not contiguous"

        self.pad_id = self.vocab["<|pad|>"]
        self.unk_id = self.vocab["<|unk|>"]
        self.eos_id = self.vocab["<|endoftext|>"]
        self.bos_id = self.eos_id
        self.mask_id = self.eos_id

        self.special_ids = {
            self.pad_id,
            self.unk_id,
            self.eos_id,
        }
        # Your tokenizer metadata
        meta = {}
        if meta_path and os.path.exists(meta_path):
            with open(meta_path, "r", encoding="utf-8") as f:
                meta = json.load(f)

        for key, val in (
            ("pad_id", self.pad_id),
            ("unk_id", self.unk_id),
            ("eos_id", self.eos_id),
            ("bos_id", self.bos_id),
            ("mask_id", self.mask_id),
        ):
            if key in meta and meta[key] != val:
                raise ValueError(
                    f"meta.json {key}={meta[key]} disagrees with vocab ({val})"
            )

        if meta.get("vocab_size") not in (None, self.vocab_size):
            log(
                f"warning: meta.json vocab_size="
                f"{meta['vocab_size']} but vocab has "
                f"{self.vocab_size}"
            )

        # Decoder-only model
        self.bos_id = self.eos_id
        self.special_ids = {
            self.pad_id,
            self.unk_id,
            self.eos_id,
        }

        self.is_end = [
            self.inv[i].endswith("</w>")
            for i in range(self.vocab_size)
        ]

        # Cache BPE results for repeated words
        self.cache = {}

    def _word_ids(self, word):
        hit = self.cache.get(word)
        if hit is not None:
            return hit

        # Character sequence + word-end marker
        syms = list(word) + ["</w>"]

        # Apply BPE merges
        while len(syms) > 1:
            best = None
            best_rank = None

            for i in range(len(syms) - 1):
                r = self.ranks.get(
                    (syms[i], syms[i + 1])
                )

                if r is not None and (
                    best_rank is None or r < best_rank
                ):
                    best = (
                        syms[i],
                        syms[i + 1]
                    )
                    best_rank = r

            if best is None:
                break

            merged = best[0] + best[1]

            out = []
            i = 0

            while i < len(syms):
                if (
                    i < len(syms) - 1
                    and syms[i] == best[0]
                    and syms[i + 1] == best[1]
                ):
                    out.append(merged)
                    i += 2
                else:
                    out.append(syms[i])
                    i += 1

            syms = out

        ids = [
            self.vocab.get(s, self.unk_id)
            for s in syms
        ]

        # Prevent unlimited cache growth
        if len(self.cache) > 3_000_000:
            self.cache.clear()

        self.cache[word] = ids
        return ids

    def encode(self, text):
        ids = []

        for w in WORD_PATTERN.findall(text.lower()):
            ids.extend(self._word_ids(w))

        return ids

    def encode_batch(self, texts):
        return [self.encode(t) for t in texts]

    def decode(self, ids):
        toks = [
            self.inv.get(int(i), "")
            for i in ids
            if int(i) not in self.special_ids
        ]

        return "".join(toks).replace("</w>", " ").strip()

class HFByteBPE:
    """byte-level BPE (tokenizer.json of legal_bpe_50k.ipynb).  No special tokens are ever added."""

    def __init__(self, path):
        from tokenizers import Tokenizer
        self.path = path
        self.tk = Tokenizer.from_file(path)
        vocab = self.tk.get_vocab()
        self.vocab_size = self.tk.get_vocab_size()
        self.pad_id, self.unk_id, self.eos_id = vocab["<|pad|>"], vocab["<|unk|>"], vocab["<|endoftext|>"]

    def encode_batch(self, texts):
        return [e.ids for e in self.tk.encode_batch(texts, add_special_tokens=False)]

def find_tokenizer_candidates(real_vocab):
    """[(kind, directory, tokenizer)] of every usable tokenizer whose vocabulary size == real_vocab"""
    roots = [C.P1_TOKENIZER_DIR] if C.P1_TOKENIZER_DIR else [INPUT_ROOT, os.getcwd()]
    cands, seen_l, seen_h = [], set(), set()
    if C.TOK_KIND in ("auto", "legalbpe"):
        for d in walk_find("vocab_50k.json", roots):
            if d in seen_l or not os.path.exists(os.path.join(d, "merges_50k.json")):
                continue
            seen_l.add(d)
            try:
                t = LegalBPE(os.path.join(d, "vocab_50k.json"), os.path.join(d, "merges_50k.json"),
                             os.path.join(d, "meta_50k.json"))
            except Exception as e:
                log(f"  skipped {d} (LegalBPE failed: {e!r})")
                continue
            if t.vocab_size == real_vocab:
                cands.append(("legalbpe", d, t))
            else:
                log(f"  skipped {d}: vocabulary {t.vocab_size} != model vocabulary {real_vocab}")
    if C.TOK_KIND in ("auto", "hf"):
        for d in walk_find("tokenizer.json", roots):
            if d in seen_h:
                continue
            seen_h.add(d)
            try:
                t = HFByteBPE(os.path.join(d, "tokenizer.json"))
            except Exception as e:
                log(f"  skipped {d}/tokenizer.json (HF tokenizers failed: {e!r})")
                continue
            if t.vocab_size == real_vocab:
                cands.append(("hf", d, t))
            else:
                log(f"  skipped {d}/tokenizer.json: vocabulary {t.vocab_size} != model vocabulary {real_vocab}")
    return cands

def decide_tokenizer(rows, strict=True, warn_at=5.5, fail_at=7.5, min_margin=0.3):
    """rows: [{'kind', 'dir', 'loss', 'n_tokens'}] -> (index of the chosen candidate, [notes]).  Raises if even the best
    candidate is hopeless (strict).  Pure function, unit-tested."""
    assert rows, "no tokenizer candidates"
    order = sorted(range(len(rows)), key=lambda i: rows[i]["loss"])
    best = order[0]
    notes = []
    if rows[best]["loss"] != rows[best]["loss"] or rows[best]["loss"] > fail_at:
        msg = (f"no candidate tokenizer fits the model: the best next-token loss is {rows[best]['loss']:.2f} nats "
               f"(a matching tokenizer gives a few nats, a mismatched one ~{math.log(50000):.0f}+)")
        if strict:
            raise RuntimeError(msg + ". Attach the tokenizer Model 3 was trained with, or set P1_TOKENIZER_DIR / TOK_KIND "
                                     "(TOK_PROBE_STRICT=False turns this error into a warning).")
        notes.append("WARNING: " + msg)
    elif rows[best]["loss"] > warn_at:
        notes.append(f"WARNING: the best next-token loss is high ({rows[best]['loss']:.2f} nats); check the tokenizer")
    if len(order) > 1 and rows[order[1]]["loss"] - rows[best]["loss"] < min_margin:
        notes.append(f"WARNING: the two best candidates are close ({rows[best]['loss']:.2f} vs {rows[order[1]]['loss']:.2f} nats): "
                     f"set P1_TOKENIZER_DIR or TOK_KIND to the tokenizer you trained Model 3 with")
    return best, notes

log("tokenizer candidates (vocabulary size must equal the model's real vocabulary, %d):" % P1_CFG.real_vocab)
TOK_CANDS = find_tokenizer_candidates(P1_CFG.real_vocab)
for _k, _d, _t in TOK_CANDS:
    log(f"  [{_k}] {_d}  (vocab {_t.vocab_size}, pad={_t.pad_id} unk={_t.unk_id} eos={_t.eos_id})")
if not TOK_CANDS:
    raise FileNotFoundError(
        "no tokenizer with a %d-id vocabulary found under /kaggle/input. Attach the tokenizer dataset Model 3 was trained with: "
        "vocab_50k.json + merges_50k.json (+ meta_50k.json), or tokenizer.json from legal_bpe_50k.ipynb "
        "(attach both if unsure - the model decides). Or set P1_TOKENIZER_DIR." % P1_CFG.real_vocab)

### 3. Phase-7 data
Queries and candidates per split; each split uses only its own pool. Anonymisation tags (`<NAME>`, `<ORG>`, `<CITATION>`) are stripped.

In [ ]:

# Prepare the data
# Keep each split's candidate pool separate.

P7_FILES = {
    "train_queries": "Phase_7_PCR/Train/train.json",
    "train_candidates": "Phase_7_PCR/Train/train_candidates.json",
    "dev_queries": "Phase_7_PCR/Dev/dev.json",
    "dev_candidates": "Phase_7_PCR/Dev/dev_candidates.json",
    "test_queries": "Phase_7_PCR/Test/test.json",
    "test_candidates": "Phase_7_PCR/Test/test_candidates.json",
}

def _looks_like_p7(path):
    p = path.lower().replace("-", "_")
    return "phase_7" in p or "phase7" in p or "pcr" in p

def p7_file(rel):
    """local copy (P7_ROOT / Kaggle input) first, then the Hugging Face dataset repo"""
    base = os.path.basename(rel)
    if C.P7_ROOT:
        for cand in (Path(C.P7_ROOT) / rel, Path(C.P7_ROOT) / base):
            if cand.is_file():
                return str(cand)
        for d in walk_find(base, [C.P7_ROOT]):
            return os.path.join(d, base)
    for d in walk_find(base, [INPUT_ROOT]):                         # an attached copy of the dataset
        if _looks_like_p7(d):
            return os.path.join(d, base)
    err = None
    try:
        from huggingface_hub import hf_hub_download
        return hf_hub_download(repo_id=C.HF_DATASET_REPO, filename=rel, repo_type="dataset", token=HF_TOKEN)
    except Exception as e:
        err = e
    raise FileNotFoundError(f"cannot find {rel}: not under P7_ROOT / {INPUT_ROOT} and the HF download failed: {err!r}")

def unwrap_json(obj):
    if isinstance(obj, dict):
        for key in ("data", "records", "examples", "queries", "candidates"):
            if isinstance(obj.get(key), (list, dict)):
                return obj[key]
    return obj

def normalize_text(value):
    if value is None:
        return ""
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, list):                                     # sentence list -> one string, order preserved
        return " ".join(str(x).strip() for x in value if x is not None).strip()
    return str(value).strip()

def records_from_queries(obj):
    """-> list of (query_id, text, [relevant ids])"""
    rows = []
    if isinstance(obj, list):
        iterator = enumerate(obj)
    elif isinstance(obj, dict):
        iterator = [(None, obj)] if any(k in obj for k in ("id", "case_id", "query_id")) else obj.items()
    else:
        raise TypeError(f"queries must be list/dict, got {type(obj)}")
    for key, rec in iterator:
        if not isinstance(rec, dict):
            continue
        cid = rec.get("id", rec.get("case_id", rec.get("query_id", key)))
        text = rec.get("text", rec.get("document", rec.get("case_text", "")))
        rel = rec.get("relevant_candidates", rec.get("relevant_candidate_ids", [])) or []
        if isinstance(rel, str):
            rel = [rel]
        rows.append((str(cid), normalize_text(text), [str(x) for x in rel]))
    return rows

def records_from_candidates(obj):
    """-> dict candidate_id -> text"""
    out = {}
    if isinstance(obj, dict):
        iterable = [(None, obj)] if any(k in obj for k in ("id", "case_id", "text", "document")) else obj.items()
    elif isinstance(obj, list):
        iterable = [(None, r) for r in obj]
    else:
        raise TypeError(f"candidates must be list/dict, got {type(obj)}")
    for key, rec in iterable:
        if isinstance(rec, dict):
            cid = rec.get("id", rec.get("case_id", rec.get("candidate_id", key)))
            txt = rec.get("text", rec.get("document", rec.get("case_text", "")))
        else:
            cid, txt = key, rec
        if cid is None:
            raise ValueError("cannot infer a candidate id - inspect the raw candidate file")
        out[str(cid)] = normalize_text(txt)
    return out

_PLACEHOLDER = re.compile(r"<\s*[A-Z][A-Z_]{1,19}\s*>")             # <NAME> <ORG> <CITATION> ...

def prep_text(t):
    if C.STRIP_PLACEHOLDERS:
        t = _PLACEHOLDER.sub(" ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t[:C.MAX_CHARS]

def limit_split(sp, max_q, max_pool, seed):
    """smoke-test helper: first `max_q` queries; pool = their gold + their own ids + random fill up to `max_pool`"""
    rng = random.Random(seed)
    qids = sp.qids[:max_q] if max_q else list(sp.qids)
    pool = set(sp.cids)
    keep = set()
    for q in qids:
        keep |= (sp.gold[q] & pool)
        if q in pool:
            keep.add(q)
    rest = [c for c in sp.cids if c not in keep]
    rng.shuffle(rest)
    n_fill = len(rest) if not max_pool else max(max_pool - len(keep), 0)
    keep |= set(rest[:n_fill])
    sp.qids = qids
    sp.cids = [c for c in sp.cids if c in keep]
    sp.qtext = {q: sp.qtext[q] for q in qids}
    sp.gold = {q: sp.gold[q] for q in qids}
    sp.ctext = {c: sp.ctext[c] for c in sp.cids}
    return sp

def load_split(name):
    q_raw = unwrap_json(json.load(open(p7_file(P7_FILES[f"{name}_queries"]), encoding="utf-8")))
    c_raw = unwrap_json(json.load(open(p7_file(P7_FILES[f"{name}_candidates"]), encoding="utf-8")))
    qrows = records_from_queries(q_raw)
    cmap = records_from_candidates(c_raw)
    sp = types.SimpleNamespace(name=name)
    sp.qids = [q for q, _, _ in qrows]
    sp.qtext = {q: prep_text(t) for q, t, _ in qrows}
    sp.gold = {q: set(r) for q, _, r in qrows}
    sp.cids = list(cmap.keys())
    sp.ctext = {c: prep_text(t) for c, t in cmap.items()}
    if C.MAX_QUERIES or C.MAX_POOL:
        sp = limit_split(sp, C.MAX_QUERIES, C.MAX_POOL, C.SEED)
        log(f"[smoke] {name}: limited to {len(sp.qids)} queries / {len(sp.cids)} candidates")
    pool = set(sp.cids)
    sp.gold_in_pool = {q: sorted((sp.gold[q] & pool) - {q}) for q in sp.qids}
    return sp

SP = {name: load_split(name) for name in ("train", "dev", "test")}

rows = []
for name, sp in SP.items():
    pool = set(sp.cids)
    ng = [len(sp.gold[q]) for q in sp.qids]
    nl = [len(sp.qtext[q].split()) for q in sp.qids]
    cl = [len(t.split()) for t in sp.ctext.values()]
    mism = sum(1 for q in sp.qids if q in sp.ctext and sp.ctext[q] != sp.qtext[q])
    rows.append({
        "split": name, "queries": len(sp.qids), "pool": len(sp.cids),
        "query_in_pool": sum(q in pool for q in sp.qids), "text_differs": mism,
        "gold_mean": round(float(np.mean(ng)), 2) if ng else 0, "gold_min": min(ng) if ng else 0,
        "gold_max": max(ng) if ng else 0,
        "gold_not_in_pool": sum(len(sp.gold[q] - pool) for q in sp.qids),
        "no_gold_queries": sum(len(sp.gold[q]) == 0 for q in sp.qids),
        "q_words_med": int(np.median(nl)) if nl else 0, "c_words_med": int(np.median(cl)) if cl else 0,
        "c_words_p95": int(np.percentile(cl, 95)) if cl else 0,
    })
log(pd.DataFrame(rows).to_string(index=False))
log("(query_in_pool > 0 means a query's own case is also in the pool: it is excluded from its own ranking)")
assert all(len(SP[s].qids) > 0 and len(SP[s].cids) > C.K for s in SP), "a split is empty / its pool is smaller than K+1"
assert sum(len(SP["train"].gold_in_pool[q]) > 0 for q in SP["train"].qids) > 0, "no train query has a gold candidate in the pool"
assert any(len(SP["dev"].gold[q]) > 0 for q in SP["dev"].qids), "dev has no labels"
RESULTS["data"] = rows
sample_txt = next(iter(SP["train"].qtext.values()))
log("sample (first 300 chars of a prepared train query):", sample_txt[:300])

### 3b. Choose the tokenizer by the model's own next-token loss

In [ ]:


@torch.inference_mode()
def probe_next_token_loss(model, tok, texts, n_tok=512, min_tok=64):
    was = model.training
    model.eval()
    tot, cnt = 0.0, 0
    for t in texts:
        ids = tok.encode_batch([t[:30000]])[0][:n_tok]
        if len(ids) < min_tok:
            continue
        x = torch.tensor([ids], dtype=torch.long, device=DEVICE)
        cos, sin = model._rope(x.size(1))
        h = model.embed(x)
        for blk in model.blocks:
            h = blk(h, cos, sin)
        logits = F.linear(model.final_norm(h), model.embed.weight).float()
        tot += float(F.cross_entropy(logits[0, :-1], x[0, 1:], reduction="sum"))
        cnt += len(ids) - 1
    model.train(was)
    return (tot / cnt if cnt else float("nan")), cnt

_dv = SP["dev"]
PROBE_TEXTS = [_dv.ctext[c] for c in _dv.cids[::max(1, len(_dv.cids) // C.TOK_PROBE_DOCS)][:C.TOK_PROBE_DOCS]]
P1_MODEL.to(DEVICE)
PROBE_ROWS = []
for _kind, _dir, _tok in TOK_CANDS:
    _loss, _n = probe_next_token_loss(P1_MODEL, _tok, PROBE_TEXTS)
    PROBE_ROWS.append({"kind": _kind, "dir": _dir, "loss": _loss, "n_tokens": _n})
log("next-token loss of the Phase-1 model under each candidate tokenizer (lower = the tokenizer the model was trained with):")
log(pd.DataFrame(PROBE_ROWS).to_string(index=False, float_format=lambda x: f"{x:.3f}"))
_best, _notes = decide_tokenizer(PROBE_ROWS, C.TOK_PROBE_STRICT)
for _n in _notes:
    log(_n)
TOK = TOK_CANDS[_best][2]
log(f">>> tokenizer: [{PROBE_ROWS[_best]['kind']}] {PROBE_ROWS[_best]['dir']} | vocab={TOK.vocab_size} pad={TOK.pad_id} "
    f"unk={TOK.unk_id} eos={TOK.eos_id} | probe loss {PROBE_ROWS[_best]['loss']:.3f} nats/token")
assert TOK.vocab_size == P1_CFG.real_vocab
assert (TOK.pad_id, TOK.unk_id) == (P1_CFG.pad_id, P1_CFG.unk_id), "special-token ids of tokenizer and checkpoint disagree"
RESULTS["tokenizer"] = {"chosen": PROBE_ROWS[_best]["kind"], "dir": PROBE_ROWS[_best]["dir"], "probe": PROBE_ROWS}
del TOK_CANDS
gc.collect()

### 4. Tokenisation and chunking of long cases

In [ ]:

# Tokenisation and chunking
# Split each case into chunks that fit the model context.

def chunk_spans(n, L, stride, max_chunks, min_tail):
    """token spans [(start, end), ...] covering a document of n tokens"""
    if n <= L:
        return [(0, n)]
    spans, s = [], 0
    while s < n and len(spans) < max_chunks:
        e = min(s + L, n)
        if spans and e - s < min_tail:                              # drop a tiny trailing chunk
            break
        spans.append((s, e))
        if e == n:
            break
        s += stride
    return spans

def _selftest_chunk_spans():
    assert chunk_spans(10, 1024, 1024, 16, 64) == [(0, 10)]
    assert chunk_spans(1024, 1024, 1024, 16, 64) == [(0, 1024)]
    assert chunk_spans(2048, 1024, 1024, 16, 64) == [(0, 1024), (1024, 2048)]
    assert chunk_spans(2048 + 30, 1024, 1024, 16, 64) == [(0, 1024), (1024, 2048)]           # tail < 64 dropped
    assert chunk_spans(2048 + 100, 1024, 1024, 16, 64) == [(0, 1024), (1024, 2048), (2048, 2148)]
    assert len(chunk_spans(10**6, 1024, 1024, 16, 64)) == 16
    assert chunk_spans(3000, 1024, 512, 16, 64)[-1][1] == 3000                                # overlap: reaches the end
    return True

_selftest_chunk_spans()

MAX_TOKENS = C.CHUNK_LEN + (C.MAX_CHUNKS - 1) * C.CHUNK_STRIDE      # hard cap on tokens kept per document

class Doc:
    __slots__ = ("ids", "spans")

    def __init__(self, ids):
        if len(ids) == 0:
            ids = np.array([TOK.unk_id], dtype=np.int32)             # never feed an empty sequence
        self.ids = ids
        self.spans = chunk_spans(len(ids), C.CHUNK_LEN, C.CHUNK_STRIDE, C.MAX_CHUNKS, C.MIN_TAIL)

_TOK_CACHE = {}

def build_docs(texts, batch=128):
    keys = [hashlib.md5(t.encode("utf-8")).hexdigest() for t in texts]
    todo = {}
    for k, t in zip(keys, texts):
        if k not in _TOK_CACHE and k not in todo:
            todo[k] = t
    items = list(todo.items())
    for i in range(0, len(items), batch):
        part = items[i:i + batch]
        enc = TOK.encode_batch([t for _, t in part])
        for (k, _), ids in zip(part, enc):
            _TOK_CACHE[k] = Doc(np.asarray(ids[:MAX_TOKENS], dtype=np.int32))
    return [_TOK_CACHE[k] for k in keys]

t0 = time.time()
for name, sp in SP.items():
    sp.qdocs = build_docs([sp.qtext[q] for q in sp.qids])
    sp.cdocs = build_docs([sp.ctext[c] for c in sp.cids])
    sp.q_index = {q: i for i, q in enumerate(sp.qids)}
    sp.c_index = {c: i for i, c in enumerate(sp.cids)}
log(f"tokenised {len(_TOK_CACHE):,} unique documents in {fmt_time(time.time() - t0)}")

rows = []
for name, sp in SP.items():
    for role, docs in (("query", sp.qdocs), ("candidate", sp.cdocs)):
        ntok = np.array([len(d.ids) for d in docs])
        nch = np.array([len(d.spans) for d in docs])
        rows.append({"split": name, "role": role, "docs": len(docs), "tok_median": int(np.median(ntok)),
                     "tok_p95": int(np.percentile(ntok, 95)), "tok_max": int(ntok.max()),
                     "chunks_mean": round(float(nch.mean()), 2), "chunks_max": int(nch.max()),
                     "at_token_cap": int((ntok >= MAX_TOKENS).sum()), "total_chunks": int(nch.sum())})
log(pd.DataFrame(rows).to_string(index=False))
RESULTS["tokenisation"] = rows

### 5. Retrieval utilities
Ranking, micro-P/R/F1@7, BM25, legal-reference overlap, dev-tuned score fusion, pseudo-relevance feedback, hard-negative mining (numpy / scikit-learn, with self-tests).

In [ ]:

# Retrieval utilities

# hard-negative mining.  Everything here has a small self-test.

def l2n(x, eps=1e-8):
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), eps)

def zrows(S):
    """per-query z-score of a score matrix [n_queries, n_candidates]"""
    S = S.astype(np.float32, copy=False)
    return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-6)

def mask_self(S, qids, cids):
    """-inf where a candidate is the query itself (queries also sit in their pool). Works on a copy."""
    S = S.astype(np.float32, copy=True)
    cidx = {c: i for i, c in enumerate(cids)}
    for i, q in enumerate(qids):
        j = cidx.get(q)
        if j is not None:
            S[i, j] = -np.inf
    return S

def rank_topk(S, qids, cids, k):
    """top-k candidate ids per query (best first), the query's own id excluded.  S: [n_queries, n_candidates]"""
    S = mask_self(S, qids, cids)
    k = min(k, S.shape[1] - 1)
    part = np.argpartition(-S, k - 1, axis=1)[:, :k]
    sc = np.take_along_axis(S, part, axis=1)
    order = np.argsort(-sc, axis=1, kind="stable")
    top = np.take_along_axis(part, order, axis=1)
    return {q: [cids[j] for j in top[i]] for i, q in enumerate(qids)}

def micro_prf(preds, gold, qids, k=7):
    """micro P / R / F1@k against the FULL gold set (no truncation, no padding)"""
    tp = n_pred = n_gold = 0
    for q in qids:
        p = list(preds[q][:k])
        g = gold[q]
        tp += len(set(p) & g)
        n_pred += len(p)
        n_gold += len(g)
    P = tp / n_pred if n_pred else 0.0
    R = tp / n_gold if n_gold else 0.0
    F1 = 2 * P * R / (P + R) if (P + R) > 0 else 0.0
    return {"P": P, "R": R, "F1": F1, "tp": tp, "n_pred": n_pred, "n_gold": n_gold}

def f1_of(S, sp, k):
    return micro_prf(rank_topk(S, sp.qids, sp.cids, k), sp.gold, sp.qids, k)["F1"]

def first_hit_rank(S, qids, cids, gold):
    """1-based rank of the best-ranked gold candidate (0 if the query has no gold in the pool)"""
    S = mask_self(S, qids, cids)
    cidx = {c: i for i, c in enumerate(cids)}
    out = []
    for i, q in enumerate(qids):
        g = [cidx[c] for c in gold[q] if c in cidx and c != q]
        out.append(int((S[i] > S[i, g].max()).sum()) + 1 if g else 0)
    return out

# BM25 ranking

def bm25_scores(q_texts, c_texts, ngram=(1, 1), k1=1.2, b=0.75):
    """Okapi BM25 with Lucene-style idf and sub-linear query term frequency.  -> [n_queries, n_candidates] float32"""
    import scipy.sparse as sps
    from sklearn.feature_extraction.text import CountVectorizer
    vec = CountVectorizer(lowercase=True, ngram_range=tuple(ngram), token_pattern=r"(?u)\b\w\w+\b",
                          dtype=np.float32, max_features=2_000_000)
    try:
        X = vec.fit_transform(c_texts).tocsr()
    except ValueError:                                              # empty vocabulary
        return np.zeros((len(q_texts), len(c_texts)), dtype=np.float32)
    N = X.shape[0]
    df = np.bincount(X.indices, minlength=X.shape[1]).astype(np.float32)
    idf = np.log(1.0 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
    dl = np.asarray(X.sum(1)).ravel()
    avgdl = max(float(dl.mean()), 1.0)
    rows = np.repeat(np.arange(N), np.diff(X.indptr))
    tf = X.data
    X.data = (tf * (k1 + 1.0) / (tf + k1 * (1.0 - b + b * dl[rows] / avgdl))).astype(np.float32)
    X = (X @ sps.diags(idf)).tocsr()
    Q = vec.transform(q_texts).tocsr()
    Q.data = (1.0 + np.log(Q.data)).astype(np.float32)
    return (Q @ X.T).toarray().astype(np.float32)

# Legal-reference matching

_NUMLIST = r"\d+[A-Za-z]?(?:\s*\(\w+\))*(?:\s*(?:,|and|&|or|to|/)\s*\d+[A-Za-z]?(?:\s*\(\w+\))*)*"
_SEC = re.compile(r"\b(?:sections?|secs?\.?|s\.)\s*(" + _NUMLIST + ")", re.I)
_ART = re.compile(r"\barticles?\s*(" + _NUMLIST + ")", re.I)
_ORD = re.compile(r"\border\s+([ivxlc]+|\d+)\s*,?\s*rules?\s*(\d+[A-Za-z]?)", re.I)
_ACT = re.compile(r"\b((?:[A-Z][A-Za-z\.&']*\s+){1,6}(?:Act|Code|Constitution))\b")
_ABBR = re.compile(r"\b(IPC|CrPC|Cr\.P\.C|CPC|C\.P\.C|NDPS|PMLA|UAPA|POCSO|TADA|MACT)\b")
_NUM = re.compile(r"\d+[A-Za-z]?")

def legal_refs(text):
    toks = []
    for m in _SEC.finditer(text):
        toks += ["sec" + n.lower() for n in _NUM.findall(m.group(1))]
    for m in _ART.finditer(text):
        toks += ["art" + n.lower() for n in _NUM.findall(m.group(1))]
    for m in _ORD.finditer(text):
        toks.append(f"ord{m.group(1).lower()}r{m.group(2).lower()}")
    for m in _ACT.finditer(text):
        name = re.sub(r"^(?:the|this|that|of|in|under|by)\s+", "", m.group(1).strip(), flags=re.I)
        name = re.sub(r"\s+", "_", name.lower())
        if len(name) > 4:
            toks.append("act:" + name)
    for m in _ABBR.finditer(text):
        toks.append("act:" + m.group(1).lower().replace(".", ""))
    return " ".join(toks)

def refs_scores(q_texts, c_texts):
    """TF-IDF cosine over the extracted legal-reference tokens only. [n_queries, n_candidates] float32"""
    from sklearn.feature_extraction.text import TfidfVectorizer
    cr = [legal_refs(t) for t in c_texts]
    qr = [legal_refs(t) for t in q_texts]
    vec = TfidfVectorizer(sublinear_tf=True, token_pattern=r"\S+", lowercase=False, min_df=1, dtype=np.float32)
    try:
        X = vec.fit_transform(cr)
    except ValueError:
        return np.zeros((len(q_texts), len(c_texts)), dtype=np.float32)
    return (vec.transform(qr) @ X.T).toarray().astype(np.float32)

# Score fusion

def fuse(comps, weights):
    out = None
    for n, w in weights.items():
        if w == 0 or n not in comps:
            continue
        out = w * comps[n] if out is None else out + w * comps[n]
    return out

DEFAULT_GRID = {"q2c": (0, 0.25, 0.5, 1, 2), "c2q": (0, 0.25, 0.5, 1, 2),
                "bm25": (0, 0.25, 0.5, 1, 2, 4, 8), "refs": (0, 0.1, 0.25, 0.5, 1)}

def tune_fusion(comps, primary, sp, k, grid=None, min_gain=0.002, passes=2):
    """
    greedy coordinate ascent on dev micro-F1@k.  `primary` keeps weight 1; every other component starts at 0 and is only
    switched on when it improves dev F1 by at least `min_gain` (guards against fitting noise on a small dev set).
    """
    grid = grid or DEFAULT_GRID
    weights = {n: 0.0 for n in comps}
    weights[primary] = 1.0
    best = base = f1_of(fuse(comps, weights), sp, k)
    for _ in range(passes):
        changed = False
        for n in comps:
            if n == primary:
                continue
            cur = weights[n]
            cand_best, cand_w = best, cur
            for w in grid.get(n, (0, 0.25, 0.5, 1, 2)):
                if w == cur:
                    continue
                trial = dict(weights)
                trial[n] = w
                f = f1_of(fuse(comps, trial), sp, k)
                if f > cand_best + (min_gain if cur == 0 else 1e-9):
                    cand_best, cand_w = f, w
            if cand_w != cur:
                weights[n], best, changed = cand_w, cand_best, True
        if not changed:
            break
    return weights, best, base

def prf_component(S, Ec, qids, cids, m):
    """
    pseudo-relevance feedback in embedding space: centroid of the m best candidates of each query (query itself excluded),
    scored against every candidate.  Relevant cases of one query tend to be mutually similar (they are co-cited).
    """
    Sm = mask_self(S, qids, cids)
    m = min(m, Sm.shape[1] - 1)
    top = np.argpartition(-Sm, m - 1, axis=1)[:, :m]
    cent = l2n(Ec[top].mean(1))
    return zrows(cent @ Ec.T)

def tune_prf(S, Ec, sp, k, ms=(3, 5, 8), gammas=(0.25, 0.5, 1.0), min_gain=0.002):
    base = f1_of(S, sp, k)
    best = (base, None, 0.0)
    for m in ms:
        P = prf_component(S, Ec, sp.qids, sp.cids, m)
        for g in gammas:
            f = f1_of(S + g * P, sp, k)
            if f > best[0]:
                best = (f, m, g)
    if best[1] is None or best[0] < base + min_gain:
        return None, 0.0, base, base
    return best[1], best[2], best[0], base

def mine_hard_negs(S, qids, cids, gold, pool_n, skip=0):
    """per query: ranks [skip, skip+pool_n) of the candidates that are neither the query itself nor gold"""
    S = mask_self(S, qids, cids)
    cidx = {c: i for i, c in enumerate(cids)}
    out = {}
    for i, q in enumerate(qids):
        for g in gold[q]:
            j = cidx.get(g)
            if j is not None:
                S[i, j] = -np.inf
        n = min(skip + pool_n, S.shape[1] - 1)
        idx = np.argpartition(-S[i], n - 1)[:n]
        idx = idx[np.argsort(-S[i, idx])][skip:]
        out[q] = [cids[j] for j in idx if np.isfinite(S[i, j])]
    return out

def per_query_table(S, sp, preds, k):
    ranks = first_hit_rank(S, sp.qids, sp.cids, sp.gold)
    rows = []
    for i, q in enumerate(sp.qids):
        g = sp.gold[q]
        hits = len(set(preds[q][:k]) & g)
        rows.append({"qid": q, "n_gold": len(g), "hits": hits,
                     "precision": hits / k, "recall": hits / len(g) if g else 0.0,
                     "first_hit_rank": ranks[i], "q_tokens": len(sp.qdocs[i].ids),
                     "q_chunks": len(sp.qdocs[i].spans)})
    return pd.DataFrame(rows)

# Self-tests
def _selftest_retrieval():
    preds = {"a": ["1", "2", "3", "4", "5", "6", "7"], "b": ["1", "2", "3", "4", "5", "6", "7"]}
    gold = {"a": {"1", "2", "9"}, "b": {"7"}}
    m = micro_prf(preds, gold, ["a", "b"], 7)
    assert (m["tp"], m["n_pred"], m["n_gold"]) == (3, 14, 4)
    assert abs(m["F1"] - 2 * (3 / 14) * (3 / 4) / (3 / 14 + 3 / 4)) < 1e-12
    S = np.array([[0.9, 0.1, 0.5, 0.8], [0.2, 0.3, 0.1, 0.95]], dtype=np.float32)
    top = rank_topk(S, ["x", "y"], ["x", "b", "c", "d"], 2)          # query "x" is in the pool at column 0
    assert top["x"] == ["d", "c"] and top["y"] == ["d", "b"], top
    # Remove the query and gold cases from hard negatives.
    cids = ["x", "g1", "n1", "n2", "n3", "n4"]
    S = np.array([[0.99, 0.95, 0.9, 0.5, 0.4, 0.3]], dtype=np.float32)
    assert mine_hard_negs(S, ["x"], cids, {"x": {"g1"}}, 2)["x"] == ["n1", "n2"]
    assert mine_hard_negs(S, ["x"], cids, {"x": {"g1"}}, 2, skip=1)["x"] == ["n2", "n3"]
    # Check BM25 and legal-reference matching.
    docs = ["tenant eviction under rent control act section 13", "murder trial under section 302 ipc",
            "property partition suit among brothers", "bail application under section 439 crpc"]
    sc = bm25_scores(["eviction of tenant rent control"], docs)[0]
    assert int(np.argmax(sc)) == 0 and sc.shape == (4,)
    r = legal_refs("Under Section 302 and 34 of the Indian Penal Code read with Article 21 of the Constitution and IPC")
    assert "sec302" in r and "sec34" in r and "art21" in r and "act:ipc" in r, r
    rs = refs_scores(["section 302 IPC murder"], docs)[0]
    assert int(np.argmax(rs)) == 1, rs
    # Check score fusion.
    rng = np.random.default_rng(0)
    nq, nc = 20, 120
    cid = [f"c{i}" for i in range(nc)]
    qid = [f"q{i}" for i in range(nq)]
    gold = {q: set(rng.choice(cid, 5, replace=False)) for q in qid}
    good = rng.normal(size=(nq, nc)).astype(np.float32)
    weak = rng.normal(size=(nq, nc)).astype(np.float32)
    for i, q in enumerate(qid):
        for g in gold[q]:
            good[i, int(g[1:])] += 3.0
            weak[i, int(g[1:])] += 0.3
    sp_ = types.SimpleNamespace(qids=qid, cids=cid, gold=gold)
    comps = {"pooled": zrows(weak), "bm25": zrows(good)}
    w, best, base = tune_fusion(comps, "pooled", sp_, 7)
    assert w["bm25"] > 0 and best > base + 0.1, (w, best, base)
    # Check pseudo-relevance feedback output.
    E = l2n(rng.normal(size=(nc, 16)).astype(np.float32))
    assert prf_component(good, E, qid, cid, 3).shape == (nq, nc)
    return True

_selftest_retrieval()

### 6. Encoder, chunk pooling, MaxSim, multi-positive loss
With CPU self-tests that run at import.

In [ ]:

# Encoder and retrieval scoring
# multi-positive contrastive loss

# Mask padding only during pooling.

class RetrievalEncoder(nn.Module):
    def __init__(self, lm, proj_dim=None):
        super().__init__()
        self.lm = lm
        d = lm.cfg.d_model
        p = proj_dim or d
        self.head = nn.Linear(d, p, bias=False)                     # identity at init -> zero-shot == plain pooling
        nn.init.eye_(self.head.weight)
        self.attn = nn.Linear(p, 1)                                 # chunk attention pooler (zero init = uniform mean)
        nn.init.zeros_(self.attn.weight)
        nn.init.zeros_(self.attn.bias)
        self.hier = None                                            # hierarchical pooler, set by the variant cell
        self.use_ckpt = False
        self.n_frozen = 0                                           # first n blocks (and the embedding) run without grad

    def hidden(self, ids):
        """final-norm hidden states [B, T, d] of the (adapted) causal LM, before the LM head"""
        lm = self.lm
        cos, sin = lm._rope(ids.size(1))
        nf = self.n_frozen
        if nf > 0:
            with torch.no_grad():
                x = lm.embed(ids)
                for blk in lm.blocks[:nf]:
                    x = blk(x, cos, sin)
        else:
            x = lm.embed(ids)
        for blk in lm.blocks[nf:]:
            if self.use_ckpt and self.training and torch.is_grad_enabled():
                x = grad_checkpoint(blk, x, cos, sin, use_reentrant=False)
            else:
                x = blk(x, cos, sin)
        return lm.final_norm(x)

    def token_features(self, ids, lens, need):
        h = self.hidden(ids).float()
        out = {}
        if "mean" in need:
            m = (torch.arange(ids.size(1), device=ids.device)[None, :] < lens[:, None]).unsqueeze(-1).float()
            out["mean"] = (h * m).sum(1) / lens[:, None].float()
        if "last" in need:
            out["last"] = h[torch.arange(ids.size(0), device=ids.device), lens - 1]
        return out

def chunk_features(enc, chunks, need, budget, train=False, use_amp=True):
    """
    chunks: list of 1-D int arrays (each <= CHUNK_LEN).  Returns {pool: [N, d] float32} in the ORIGINAL chunk order.
    Chunks are sorted by length so micro-batches carry little padding; each micro-batch holds <= `budget` tokens.
    train=True keeps the autograd graph (features stay on the GPU), otherwise features are moved to the CPU.
    """
    n = len(chunks)
    lens = np.array([len(c) for c in chunks], dtype=np.int64)
    order = np.argsort(-lens, kind="stable")
    outs = {k: [] for k in need}
    i = 0
    while i < n:
        L = int(lens[order[i]])
        mb = max(1, budget // max(L, 1))
        sel = order[i:i + mb]
        ids = np.zeros((len(sel), L), dtype=np.int64)
        for r, j in enumerate(sel):
            ids[r, :lens[j]] = chunks[j]
        ids_t = torch.from_numpy(ids).to(DEVICE)
        lens_t = torch.from_numpy(lens[sel]).to(DEVICE)
        with (amp_ctx() if use_amp else torch.autocast(device_type="cuda", enabled=False)):
            feats = enc.token_features(ids_t, lens_t, need)
        for k in need:
            outs[k].append(feats[k] if train else feats[k].float().cpu())
        i += mb
    inv = np.empty(n, dtype=np.int64)
    inv[order] = np.arange(n)
    res = {}
    for k in need:
        cat = torch.cat(outs[k], dim=0)
        res[k] = cat[torch.from_numpy(inv).to(cat.device)]
    return res

def docs_to_chunks(docs, train_chunks=None, rng=None):
    """
    flatten documents into chunks.  seg[i] = document of chunk i (non-decreasing), cpos[i] = index of the chunk inside
    its document.  train_chunks: keep only that many randomly chosen chunks per document (training).
    """
    chunks, seg, cpos = [], [], []
    for di, d in enumerate(docs):
        idxs = list(range(len(d.spans)))
        if train_chunks and len(idxs) > train_chunks:
            idxs = sorted(rng.choice(len(idxs), train_chunks, replace=False).tolist())
        for ci in idxs:
            s, e = d.spans[ci]
            chunks.append(d.ids[s:e])
            seg.append(di)
            cpos.append(ci)
    return chunks, np.asarray(seg, dtype=np.int64), np.asarray(cpos, dtype=np.int64)

def chunk_vectors(enc, feat):
    return F.normalize(enc.head(feat.float()), dim=-1)

def pool_chunks(z, seg, n_docs, doc_pool, attn=None):
    """unit chunk vectors [N, p] -> one unit vector per document by pooling its chunks.  Differentiable."""
    p = z.size(1)
    if doc_pool == "first":
        counts = torch.bincount(seg, minlength=n_docs)
        starts = torch.cumsum(counts, 0) - counts
        out = z[starts]
    elif doc_pool == "max":
        idx = seg[:, None].expand(-1, p).contiguous()
        out = z.new_full((n_docs, p), float("-inf")).scatter_reduce(0, idx, z, reduce="amax", include_self=False)
    elif doc_pool == "attn":
        s = attn(z).squeeze(-1)
        w = torch.exp(s - s.max().detach())
        out = z.new_zeros(n_docs, p).index_add(0, seg, z * w[:, None])   # the per-doc denominator cancels in the L2 norm
    else:                                                           # "mean" (sum of unit vectors; L2 norm makes it a mean)
        out = z.new_zeros(n_docs, p).index_add(0, seg, z)
    return F.normalize(out, dim=-1)

def doc_vectors(enc, feat, seg, cpos, n_docs):
    """training path: chunk features -> (document vectors [n_docs, p], chunk vectors z [N, p])"""
    z = chunk_vectors(enc, feat)
    if C.DOC_MODE == "hier":
        return F.normalize(enc.hier(z, seg, cpos, n_docs), dim=-1), z
    return pool_chunks(z, seg, n_docs, C.DOC_POOL, enc.attn), z

def doc_vectors_blocked(enc, feat, seg_np, cpos_np, n_docs, blk=512):
    """inference path.  The hierarchical pooler pads every document to the longest one, so it runs in document blocks."""
    z = chunk_vectors(enc, feat)
    if C.DOC_MODE != "hier":
        return pool_chunks(z, torch.from_numpy(seg_np).to(z.device), n_docs, C.DOC_POOL, enc.attn), z
    outs = []
    for d0 in range(0, n_docs, blk):
        d1 = min(d0 + blk, n_docs)
        c0 = int(np.searchsorted(seg_np, d0, "left"))
        c1 = int(np.searchsorted(seg_np, d1, "left"))
        seg_b = torch.from_numpy(seg_np[c0:c1] - d0).to(z.device)
        cpos_b = torch.from_numpy(cpos_np[c0:c1]).to(z.device)
        outs.append(F.normalize(enc.hier(z[c0:c1], seg_b, cpos_b, d1 - d0), dim=-1))
    return torch.cat(outs, 0), z

# Chunk-level MaxSim scoring.

def maxsim_matrix(zq, segq, nq, zc, segc, nc):
    sim = zq @ zc.T                                                  # [Nq, Nc] chunk-chunk cosine
    Nq, Nc = sim.shape
    idx = segc[None, :].expand(Nq, Nc).contiguous()
    qc = sim.new_full((Nq, nc), float("-inf")).scatter_reduce(1, idx, sim, reduce="amax", include_self=False)
    cnt_q = torch.bincount(segq, minlength=nq).clamp(min=1).float()
    q2c = qc.new_zeros(nq, nc).index_add(0, segq, qc) / cnt_q[:, None]
    idx2 = segq[:, None].expand(Nq, Nc).contiguous()
    cq = sim.new_full((nq, Nc), float("-inf")).scatter_reduce(0, idx2, sim, reduce="amax", include_self=False)
    cnt_c = torch.bincount(segc, minlength=nc).clamp(min=1).float()
    c2q = cq.new_zeros(nq, nc).index_add(1, segc, cq) / cnt_c[None, :]
    return q2c, c2q

@torch.inference_mode()
def maxsim_eval(z, raw, blk=32):
    """MaxSim score matrices [n_queries, n_candidates] (numpy) of a whole split; queries are processed in blocks"""
    seg, nq, nc = raw["seg"], raw["nq"], raw["n_docs"] - raw["nq"]
    nqc = int(np.searchsorted(seg, nq, "left"))
    zq, zc = z[:nqc], z[nqc:]
    segq = seg[:nqc]
    segc = torch.from_numpy(seg[nqc:] - nq).to(z.device)
    q2c = np.zeros((nq, nc), np.float32)
    c2q = np.zeros((nq, nc), np.float32)
    for b0 in range(0, nq, blk):
        b1 = min(b0 + blk, nq)
        lo = int(np.searchsorted(segq, b0, "left"))
        hi = int(np.searchsorted(segq, b1, "left"))
        sq = torch.from_numpy(segq[lo:hi] - b0).to(z.device)
        a, b = maxsim_matrix(zq[lo:hi], sq, b1 - b0, zc, segc, nc)
        q2c[b0:b1] = a.cpu().numpy()
        c2q[b0:b1] = b.cpu().numpy()
    return q2c, c2q

def multipos_loss(S, pos_mask, neg_mask):
    """
    multi-positive InfoNCE.  S: [B, M] logits (already divided by the temperature); pos_mask / neg_mask: bool [B, M].
    every positive is contrasted with all negatives of its query:  -log( e^{s_p} / (e^{s_p} + sum_n e^{s_n}) )
    other gold cases and the query itself are in neither mask.
    """
    neg_lse = torch.logsumexp(S.masked_fill(~neg_mask, -1e4), dim=1, keepdim=True)
    per = F.softplus(neg_lse - S)
    return (per * pos_mask).sum() / pos_mask.sum().clamp(min=1)

# self-tests (CPU, run at import)
def _selftest_core():
    g = torch.Generator().manual_seed(0)
    d = 8
    attn = nn.Linear(d, 1)
    nn.init.normal_(attn.weight, std=0.5)
    feat = torch.randn(7, d, generator=g)
    seg = torch.tensor([0, 0, 0, 1, 2, 2, 3])                       # 4 docs with 3 / 1 / 2 / 1 chunks
    z = F.normalize(feat, dim=-1)
    for pool in ("mean", "max", "first", "attn"):
        got = pool_chunks(z, seg, 4, pool, attn)
        for di in range(4):
            zz = z[seg == di]
            ref = {"mean": lambda: zz.mean(0), "max": lambda: zz.max(0).values, "first": lambda: zz[0],
                   "attn": lambda: (torch.softmax(attn(zz).squeeze(-1), 0)[:, None] * zz).sum(0)}[pool]()
            assert torch.allclose(got[di], F.normalize(ref, dim=-1), atol=1e-5), (pool, di)
    # MaxSim == naive loops
    zq = F.normalize(torch.randn(5, d, generator=g), dim=-1)
    zc = F.normalize(torch.randn(9, d, generator=g), dim=-1)
    segq = torch.tensor([0, 0, 1, 1, 1])
    segc = torch.tensor([0, 1, 1, 2, 2, 2, 2, 3, 3])
    q2c, c2q = maxsim_matrix(zq, segq, 2, zc, segc, 4)
    for i in range(2):
        for j in range(4):
            sim = zq[segq == i] @ zc[segc == j].T
            assert abs(float(q2c[i, j]) - float(sim.max(1).values.mean())) < 1e-5
            assert abs(float(c2q[i, j]) - float(sim.max(0).values.mean())) < 1e-5
    # maxsim_eval (blocked) == one-shot
    raw = {"seg": np.array([0, 0, 1, 1, 1, 2, 3, 3, 4, 4, 4, 4, 5, 6, 6]), "nq": 2, "n_docs": 7}   # 2 queries + 5 candidates
    zz = F.normalize(torch.randn(15, d, generator=g), dim=-1)
    a1, b1 = maxsim_eval(zz, raw, blk=1)
    a2, b2 = maxsim_eval(zz, raw, blk=32)
    assert np.allclose(a1, a2, atol=1e-6) and np.allclose(b1, b2, atol=1e-6) and a1.shape == (2, 5)
    # multi-positive loss == naive
    S = torch.randn(3, 6, generator=g)
    pos = torch.zeros(3, 6, dtype=torch.bool)
    neg = torch.ones(3, 6, dtype=torch.bool)
    for i, ps in enumerate(([0, 1], [2], [3, 4, 5])):
        for p_ in ps:
            pos[i, p_] = True
            neg[i, p_] = False
    got = multipos_loss(S, pos, neg)
    ref, cnt = 0.0, 0
    for i in range(3):
        for j in range(6):
            if pos[i, j]:
                den = torch.exp(S[i, j]) + torch.exp(S[i][neg[i]]).sum()
                ref += float(-torch.log(torch.exp(S[i, j]) / den))
                cnt += 1
    assert abs(float(got) - ref / cnt) < 1e-5, (float(got), ref / cnt)
    # position-interpolated RoPE with scale 1 == the Phase-1 tables
    c1, s1 = build_rope(64, 128)
    c2, s2 = rope_tables(64, 128, 1.0)
    assert torch.allclose(c1, c2) and torch.allclose(s1, s2)
    c4, _ = rope_tables(64, 256, 4.0)
    assert torch.allclose(c4[::4][:64], c1[:64], atol=1e-6)         # position 4p at scale 4 == position p at scale 1
    # chunking helper
    docs = [types.SimpleNamespace(ids=np.arange(300), spans=[(0, 100), (100, 200), (200, 300)]),
            types.SimpleNamespace(ids=np.arange(50), spans=[(0, 50)])]
    ch, sg, cp = docs_to_chunks(docs)
    assert [len(c) for c in ch] == [100, 100, 100, 50] and sg.tolist() == [0, 0, 0, 1] and cp.tolist() == [0, 1, 2, 0]
    ch, sg, cp = docs_to_chunks(docs, 2, np.random.default_rng(0))
    assert len(ch) == 3 and sg.tolist() == [0, 0, 1] and len(set(cp[:2].tolist())) == 2
    return True

_selftest_core()
log("self-tests passed: retrieval utilities, chunk pooling, MaxSim, multi-positive loss, RoPE interpolation")

def build_encoder():
    lm = P1_MODEL.to(DEVICE)
    for p in lm.parameters():
        p.requires_grad_(False)
    if C.ROPE_PI:
        extend_rope(lm, C.CHUNK_LEN + 64, C.CHUNK_LEN / 1024.0)
        log(f"RoPE position interpolation: {C.CHUNK_LEN}-token chunks, positions divided by {C.CHUNK_LEN / 1024.0:g}")
    enc = RetrievalEncoder(lm, C.PROJ_DIM).to(DEVICE)
    enc.use_ckpt = bool(C.GRAD_CKPT)
    return enc

def trainable_state(enc):
    return {n: p.detach().cpu().clone() for n, p in enc.named_parameters() if p.requires_grad}

def load_trainable_state(enc, state):
    missing, unexpected = enc.load_state_dict(state, strict=False)
    assert not unexpected, f"unexpected keys in the checkpoint: {unexpected[:5]}"
    own = {n for n, p in enc.named_parameters() if p.requires_grad}
    assert own <= set(state), f"checkpoint lacks trainable params: {sorted(own - set(state))[:5]}"

def param_report(enc):
    total = sum(p.numel() for p in enc.parameters())
    train = sum(p.numel() for p in enc.parameters() if p.requires_grad)
    groups = defaultdict(int)
    for n, p in enc.named_parameters():
        if p.requires_grad:
            if n.startswith("lm.blocks."):
                kind = "adapter (LoRMA+ A/B)" if (n.endswith(".A") or n.endswith(".B")) else "unfrozen Phase-1 block weights"
            elif n.startswith("lm.embed"):
                kind = "token embedding"
            elif n.startswith("lm."):
                kind = "final norm"
            elif n.startswith("head."):
                kind = "projection head"
            elif n.startswith("hier."):
                kind = "hierarchical chunk pooler"
            else:
                kind = "chunk attention pooler"
            groups[kind] += p.numel()
    return {"total_params": total, "trainable_params": train, "trainable_percent": 100.0 * train / total,
            "frozen_params": total - train, "trainable_by_group": dict(groups)}

@torch.inference_mode()
def probe_vectors(enc, docs, n=4, max_len=256):
    """first-chunk mean features of a few documents, in full fp32 (used for the no-op checks)"""
    was = enc.training
    enc.eval()
    ch = [d.ids[:max_len] for d in docs[:n]]
    f = chunk_features(enc, ch, ("mean",), 8192, train=False, use_amp=False)["mean"]
    enc.train(was)
    return f

@torch.inference_mode()
def encode_split_raw(enc, sp, need=("mean", "last")):
    """one pass of the LM over queries + pool of a split (all chunks).  Keeps the raw token-pooled chunk features."""
    was = enc.training
    enc.eval()
    docs = sp.qdocs + sp.cdocs
    chunks, seg, cpos = docs_to_chunks(docs)
    t0 = time.time()
    feats = chunk_features(enc, chunks, need, C.TOK_BUDGET_EVAL, train=False)
    enc.train(was)
    return {"feats": feats, "seg": seg, "cpos": cpos, "n_docs": len(docs), "nq": len(sp.qdocs),
            "seconds": time.time() - t0, "n_chunks": len(chunks)}

def _split_center(E, nq, center):
    Qe, Ce = E[:nq], E[nq:]
    if center == "pool":                                            # unsupervised: statistics of the evaluated pool only
        mu = Ce.mean(0, keepdims=True)
        Qe, Ce = l2n(Qe - mu), l2n(Ce - mu)
    return Qe, Ce

@torch.inference_mode()
def embed_from_raw(enc, raw, token_pool, doc_pool, center="none"):
    """simple pooled embeddings with arbitrary pooling settings (used for the zero-shot grid) -> numpy (Q, C)"""
    f = raw["feats"][token_pool].to(DEVICE)
    z = chunk_vectors(enc, f)
    E = pool_chunks(z, torch.from_numpy(raw["seg"]).to(DEVICE), raw["n_docs"], doc_pool, enc.attn).float().cpu().numpy()
    return _split_center(E, raw["nq"], center)

@torch.inference_mode()
def embed_model(enc, raw, center="none"):
    """document embeddings of the MODEL AS CONFIGURED (pooled or hierarchical) + chunk vectors z -> (Qe, Ce, z)"""
    f = raw["feats"][C.TOKEN_POOL].to(DEVICE)
    D, z = doc_vectors_blocked(enc, f, raw["seg"], raw["cpos"], raw["n_docs"])
    Qe, Ce = _split_center(D.float().cpu().numpy(), raw["nq"], center)
    return Qe, Ce, z

def model_components(enc, raw):
    """score matrices of the trained model for one split:  pooled document cosine (+ MaxSim directions)"""
    Qe, Ce, z = embed_model(enc, raw)
    comps = {"pooled": Qe @ Ce.T}
    if C.MAXSIM:
        comps["q2c"], comps["c2q"] = maxsim_eval(z, raw)
    return comps, Qe, Ce

def model_score(comps):
    """default combination used for model selection: z(pooled) [+ 0.5 z(q2c) + 0.5 z(c2q)]"""
    w = {"pooled": 1.0}
    if "q2c" in comps:
        w.update(q2c=0.5, c2q=0.5)
    return sum(wt * zrows(comps[k]) for k, wt in w.items()) / sum(w.values())

def evaluate_dense(Qe, Ce, sp, k=None):
    k = k or C.K
    S = Qe @ Ce.T
    preds = rank_topk(S, sp.qids, sp.cids, k)
    return micro_prf(preds, sp.gold, sp.qids, k), preds, S

### 6b. The adaptation technique: LoRMA+

In [ ]:

# Model adaptation

class LoRMAPlusLinear(nn.Module):
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        out, dev = base.out_features, base.weight.device
        self.A = nn.Parameter(torch.empty(r, out, device=dev))
        self.B = nn.Parameter(torch.zeros(out, r, device=dev))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        y = self.base(x)                                            # W0 x
        return y + F.linear(F.linear(self.drop(y), self.A), self.B) * self.scale

def inject_lormaplus(model, r, alpha, dropout, targets):
    n = 0
    for blk in model.blocks:
        for parent in (blk.attn, blk.ffn):
            for name in targets:
                lin = getattr(parent, name, None)
                if isinstance(lin, nn.Linear):
                    setattr(parent, name, LoRMAPlusLinear(lin, r, alpha, dropout))
                    n += 1
    return n

def setup_adaptation(enc):
    n = inject_lormaplus(enc.lm, C.LORMA_R, C.LORMA_ALPHA, C.LORMA_DROPOUT, C.LORMA_TARGETS)
    for name, p in enc.named_parameters():
        p.requires_grad_((name.startswith("lm.") and (name.endswith(".A") or name.endswith(".B"))) or name.startswith("head."))
    for p in enc.attn.parameters():
        p.requires_grad_(C.DOC_MODE == "pooled" and C.DOC_POOL == "attn")
    enc.n_frozen = 0
    return {"technique": f"LoRMA+ r={C.LORMA_R} alpha={C.LORMA_ALPHA} on {len(C.LORMA_TARGETS)} linears/block", "modules": n}

def optimizer_groups(enc):
    adapters = [p for n, p in enc.named_parameters() if p.requires_grad and n.startswith("lm.")]
    rest = [p for n, p in enc.named_parameters() if p.requires_grad and not n.startswith("lm.")]
    return [{"params": adapters, "lr": C.LR, "weight_decay": 0.0},
            {"params": rest, "lr": C.HEAD_LR, "weight_decay": C.WEIGHT_DECAY}]

### 7. Baselines, zero-shot Phase 1, apply the adaptation, parameter report, hard negatives

In [ ]:

# Build the encoder
# hard negatives

ENC = build_encoder()
N_BASE_PARAMS = sum(p.numel() for p in ENC.lm.parameters())
dev = SP["dev"]

def dev_grid(enc, raw, sp, token_pools=("mean", "last"), doc_pools=("mean", "max", "first"), centers=("none", "pool")):
    rows = []
    for tp in token_pools:
        for dp in doc_pools:
            for ce in centers:
                Qe, Ce = embed_from_raw(enc, raw, tp, dp, ce)
                m, _, _ = evaluate_dense(Qe, Ce, sp)
                rows.append({"token_pool": tp, "doc_pool": dp, "center": ce,
                             "P@7": round(m["P"], 4), "R@7": round(m["R"], 4), "F1@7": round(m["F1"], 4)})
    return pd.DataFrame(rows).sort_values("F1@7", ascending=False).reset_index(drop=True)

# lexical baselines on dev (classical; no pretrained model)
t0 = time.time()
BASE = {}
if C.USE_BM25:
    S_bm25_dev = bm25_scores([dev.qtext[q] for q in dev.qids], [dev.ctext[c] for c in dev.cids], C.BM25_NGRAM, C.BM25_K1, C.BM25_B)
    BASE["bm25"] = micro_prf(rank_topk(S_bm25_dev, dev.qids, dev.cids, C.K), dev.gold, dev.qids, C.K)
if C.USE_REFS:
    S_refs_dev = refs_scores([dev.qtext[q] for q in dev.qids], [dev.ctext[c] for c in dev.cids])
    BASE["legal_refs"] = micro_prf(rank_topk(S_refs_dev, dev.qids, dev.cids, C.K), dev.gold, dev.qids, C.K)
for k_, m_ in BASE.items():
    log(f"[baseline] {k_:10s} dev  P@7={m_['P']:.4f}  R@7={m_['R']:.4f}  F1@7={m_['F1']:.4f}")
log(f"(lexical baselines took {fmt_time(time.time() - t0)})")
RESULTS["lexical_dev"] = BASE

# checks on the untouched Phase-1 model + zero-shot pooling grid
PROBE_DOCS = dev.cdocs

@torch.inference_mode()
def padding_check(enc, docs):
    """right-padded, length-sorted micro-batches must give the same chunk features as encoding every chunk alone"""
    ch = [d.ids[:L] for d, L in zip(docs[:4], (200, 37, 120, 64))]
    was = enc.training
    enc.eval()
    batch = chunk_features(enc, ch, ("mean", "last"), 8192, train=False, use_amp=False)
    worst = 0.0
    for i, c in enumerate(ch):
        one = chunk_features(enc, [c], ("mean", "last"), 8192, train=False, use_amp=False)
        for k in ("mean", "last"):
            worst = max(worst, float((batch[k][i] - one[k][0]).abs().max()))
    enc.train(was)
    assert worst < 1e-3, f"padding / ordering changes the chunk features (max abs diff {worst:.2e})"
    return worst

log(f"padding + ordering check passed (max abs diff {padding_check(ENC, PROBE_DOCS):.1e})")
V0 = probe_vectors(ENC, PROBE_DOCS)
if C.RUN_ZERO_SHOT:
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    raw0 = encode_split_raw(ENC, dev)
    log(f"[zero-shot] encoded dev: {raw0['n_docs']:,} docs / {raw0['n_chunks']:,} chunks in "
        f"{fmt_time(raw0['seconds'])} ({raw0['n_docs'] / max(raw0['seconds'], 1e-9):.1f} docs/s)")
    GRID0 = dev_grid(ENC, raw0, dev)
    log("[zero-shot] Phase-1 Model 2, dev, no adaptation (token pooling x document pooling x centring):")
    log(GRID0.to_string(index=False))
    RESULTS["zero_shot_grid"] = GRID0.to_dict("records")
    RESULTS["efficiency_zero_shot"] = {"dev_docs": raw0["n_docs"], "dev_chunks": raw0["n_chunks"], "seconds": raw0["seconds"],
                                       "docs_per_s": raw0["n_docs"] / max(raw0["seconds"], 1e-9),
                                       "peak_gpu_mem_gb": (torch.cuda.max_memory_allocated() / 2 ** 30) if DEVICE.type == "cuda" else None}
    del raw0
    gc.collect()

# apply the adaptation technique of this notebook
ADAPT = setup_adaptation(ENC)
V1 = probe_vectors(ENC, PROBE_DOCS)
noop = float((V1 - V0).abs().max())
assert noop < 1e-5, f"the adaptation changed the Phase-1 output at init (max abs diff {noop:.3e}) - it must start at Phase 1"
PR = param_report(ENC)
RESULTS["params"] = {**PR, "base_phase1_params": N_BASE_PARAMS, "adaptation": ADAPT["technique"]}
log(f"\n>>> ADAPTATION: {ADAPT['technique']}  (no-op check at init passed, diff {noop:.1e})")
log(f">>> PARAMETERS: total {PR['total_params'] / 1e6:.2f}M | trainable {PR['trainable_params'] / 1e6:.3f}M = "
    f"{PR['trainable_percent']:.2f}% | frozen {PR['frozen_params'] / 1e6:.2f}M")
for k_, v_ in PR["trainable_by_group"].items():
    log(f"      {k_:34s} {v_ / 1e6:8.3f}M  ({100 * v_ / PR['total_params']:.2f}% of all parameters)")

# BM25 hard negatives for the training queries
tr = SP["train"]
t0 = time.time()
S_bm25_tr = bm25_scores([tr.qtext[q] for q in tr.qids], [tr.ctext[c] for c in tr.cids], (1, 1), C.BM25_K1, C.BM25_B)
m_tr = micro_prf(rank_topk(S_bm25_tr, tr.qids, tr.cids, C.K), tr.gold, tr.qids, C.K)
HARD_NEGS_BM25 = mine_hard_negs(S_bm25_tr, tr.qids, tr.cids, tr.gold, C.HARD_POOL)
HARD_NEGS_MAP = {q: list(v) for q, v in HARD_NEGS_BM25.items()}
log(f"[train] BM25 F1@7={m_tr['F1']:.4f}; hard-negative pools mined for {len(HARD_NEGS_MAP)} queries "
    f"(median {int(np.median([len(v) for v in HARD_NEGS_MAP.values()]))} per query) in {fmt_time(time.time() - t0)}")
RESULTS["bm25_train"] = m_tr
del S_bm25_tr
gc.collect()

### 8. Training (resumable)

In [ ]:

# Training

# Checkpoints are saved after each epoch.

LATEST, BEST = os.path.join(CKPT_DIR, "p7_latest.pt"), os.path.join(CKPT_DIR, "p7_best.pt")
LATEST_META, BEST_META = os.path.join(CKPT_DIR, "p7_latest_meta.json"), os.path.join(CKPT_DIR, "p7_best_meta.json")
LOG_CSV = os.path.join(CKPT_DIR, "p7_train_log.csv")

def _gscaler():
    enabled = AMP["enabled"] and AMP["dtype"] == torch.float16
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled, init_scale=2.0 ** 12)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=enabled, init_scale=2.0 ** 12)

def lr_scale(step, total, warm):
    if step < warm:
        return (step + 1) / max(warm, 1)
    prog = (step - warm) / max(1, total - warm)
    return C.MIN_LR_RATIO + (1 - C.MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * prog))

def make_optimizer():
    groups = optimizer_groups(ENC)
    for g in groups:
        g["base_lr"] = g["lr"]
    params = [p for g in groups for p in g["params"]]
    assert len({id(p) for p in params}) == len(params), "a parameter appears in two optimizer groups"
    assert {id(p) for p in params} == {id(p) for p in ENC.parameters() if p.requires_grad}, \
        "optimizer groups do not cover exactly the trainable parameters"
    return torch.optim.AdamW(groups, betas=(0.9, 0.999)), params

def embed_train(docs, rng):
    """documents -> (document vectors, chunk vectors, chunk->document map), with autograd"""
    chunks, seg, cpos = docs_to_chunks(docs, C.TRAIN_CHUNKS, rng)
    f = chunk_features(ENC, chunks, (C.TOKEN_POOL,), C.TOK_BUDGET_TRAIN, train=True)[C.TOKEN_POOL]
    seg_t, cpos_t = torch.from_numpy(seg).to(DEVICE), torch.from_numpy(cpos).to(DEVICE)
    D, z = doc_vectors(ENC, f, seg_t, cpos_t, len(docs))
    return D, z, seg_t

def evaluate_dev_now():
    raw = encode_split_raw(ENC, SP["dev"], need=(C.TOKEN_POOL,))
    comps, _, _ = model_components(ENC, raw)
    S = model_score(comps)
    m = micro_prf(rank_topk(S, SP["dev"].qids, SP["dev"].cids, C.K), SP["dev"].gold, SP["dev"].qids, C.K)
    m["F1_pooled"] = f1_of(comps["pooled"], SP["dev"], C.K)
    return m, raw["seconds"]

def find_resume():
    best = None
    for d in walk_find("p7_latest_meta.json", [CKPT_DIR, INPUT_ROOT]):
        meta = read_json(os.path.join(d, "p7_latest_meta.json"))
        if not meta or meta.get("tag") != C.P7_TAG or not os.path.exists(os.path.join(d, "p7_latest.pt")):
            continue
        if best is None or meta["epoch"] > best[1]["epoch"]:
            best = (d, meta)
    return best

def save_best(epoch, f1):
    atomic_torch_save({"trainable": trainable_state(ENC), "tag": C.P7_TAG, "dev_f1": f1, "epoch": epoch}, BEST)
    atomic_json({"tag": C.P7_TAG, "dev_f1": f1, "epoch": epoch}, BEST_META)

# Stage 0
def run_ssl():
    tr = SP["train"]
    seen, pool = set(), []
    for d in tr.qdocs + tr.cdocs:                                    # unique documents with >= 2 chunks
        if id(d) not in seen and len(d.spans) >= 2:
            seen.add(id(d))
            pool.append(d)
    B = C.SSL_BATCH
    if len(pool) < 2 * B:
        log(f"SSL skipped: only {len(pool)} train documents have >= 2 chunks")
        return
    steps_per_epoch = len(pool) // B
    total = steps_per_epoch * C.SSL_EPOCHS
    warm = max(1, int(0.1 * total))
    opt, params = make_optimizer()
    scaler = _gscaler()
    ENC.train()
    gs, t0, run = 0, time.time(), []
    rng = np.random.default_rng(C.SEED + 777)
    log(f"[SSL] {len(pool)} train documents | {steps_per_epoch} steps/epoch x {C.SSL_EPOCHS} epochs | batch {B} pairs")
    for epoch in range(C.SSL_EPOCHS):
        perm = rng.permutation(len(pool))
        for step in range(steps_per_epoch):
            a_chunks, b_chunks = [], []
            for i in perm[step * B:(step + 1) * B]:
                d = pool[i]
                ci, cj = rng.choice(len(d.spans), 2, replace=False)
                a_chunks.append(d.ids[slice(*d.spans[int(ci)])])
                b_chunks.append(d.ids[slice(*d.spans[int(cj)])])
            f = chunk_features(ENC, a_chunks + b_chunks, (C.TOKEN_POOL,), C.TOK_BUDGET_TRAIN, train=True)[C.TOKEN_POOL]
            z = chunk_vectors(ENC, f)
            logits = (z[:B] @ z[B:].T) / C.TEMP
            tgt = torch.arange(B, device=DEVICE)
            loss = 0.5 * (F.cross_entropy(logits, tgt) + F.cross_entropy(logits.T, tgt))
            sc = lr_scale(gs, total, warm)
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * C.SSL_LR_SCALE * sc
            if not torch.isfinite(loss):
                opt.zero_grad(set_to_none=True)
                gs += 1
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, C.GRAD_CLIP)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            gs += 1
            run.append(float(loss))
            if gs % 10 == 0 or gs == total:
                log(f"[SSL ep {epoch + 1}/{C.SSL_EPOCHS}] step {step + 1}/{steps_per_epoch} | loss {np.mean(run[-10:]):.4f} | "
                    f"elapsed {fmt_time(time.time() - t0)}")

# Stage 1
def refresh_hard_negs():
    """re-mine hard negatives with the CURRENT model on the train pool; mixed with the BM25 ones (ranks skipped: possible false negatives)"""
    tr = SP["train"]
    t0 = time.time()
    raw = encode_split_raw(ENC, tr, need=(C.TOKEN_POOL,))
    Qe, Ce, _ = embed_model(ENC, raw)
    S = Qe @ Ce.T
    m = micro_prf(rank_topk(S, tr.qids, tr.cids, C.K), tr.gold, tr.qids, C.K)
    dense = mine_hard_negs(S, tr.qids, tr.cids, tr.gold, C.HARD_POOL, skip=C.DENSE_SKIP)
    for q in tr.qids:
        mixed, seen = [], set()
        a, b = HARD_NEGS_BM25.get(q, []), dense.get(q, [])
        for i in range(max(len(a), len(b))):
            for lst in (a, b):
                if i < len(lst) and lst[i] not in seen:
                    seen.add(lst[i])
                    mixed.append(lst[i])
        HARD_NEGS_MAP[q] = mixed
    log(f"  >> hard negatives refreshed with the model (train F1@7 of the current model {m['F1']:.4f}) in {fmt_time(time.time() - t0)}")
    del raw, Qe, Ce, S
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

def run_training():
    tr = SP["train"]
    train_qs = [q for q in tr.qids if tr.gold_in_pool[q]]
    B = C.BATCH_Q
    steps_per_epoch = math.ceil(len(train_qs) / B)
    total_steps = steps_per_epoch * C.EPOCHS
    warm = max(1, int(C.WARMUP_FRAC * total_steps))
    log(f"train queries with gold in the pool: {len(train_qs)}/{len(tr.qids)} | {steps_per_epoch} steps/epoch x "
        f"{C.EPOCHS} epochs = {total_steps} steps | warm-up {warm}")

    state = {"epoch": 0, "gs": 0, "best_f1": -1.0, "best_epoch": -1, "bad": 0, "history": [], "hard_negs": None}
    prev = find_resume()
    if prev is None and C.SSL_EPOCHS > 0:
        run_ssl()
    opt, all_trainable = make_optimizer()
    scaler = _gscaler()
    if prev is not None:
        d, meta = prev
        pl = torch.load(os.path.join(d, "p7_latest.pt"), map_location="cpu", weights_only=False)
        load_trainable_state(ENC, pl["trainable"])
        opt.load_state_dict(pl["opt"])
        scaler.load_state_dict(pl["scaler"])
        state = pl["state"]
        if state.get("hard_negs"):
            HARD_NEGS_MAP.update(state["hard_negs"])
        if os.path.abspath(d) != os.path.abspath(CKPT_DIR) and os.path.exists(os.path.join(d, "p7_best.pt")):
            shutil.copy(os.path.join(d, "p7_best.pt"), BEST)
            shutil.copy(os.path.join(d, "p7_best_meta.json"), BEST_META)
        log(f"RESUMED from {d}: {state['epoch']} epochs done, best dev F1@7 {state['best_f1']:.4f} (epoch {state['best_epoch']})")

    if state["best_f1"] < 0:                                          # epoch 0 = Phase 1 (+ stage-0 result)
        m0, _ = evaluate_dev_now()
        state.update(best_f1=m0["F1"], best_epoch=0)
        save_best(0, m0["F1"])
        state["history"].append({"epoch": 0, "train_loss": None, "dev_P": m0["P"], "dev_R": m0["R"], "dev_F1": m0["F1"],
                                 "dev_F1_pooled": m0["F1_pooled"]})
        log(f"epoch 0 (start of supervised training)  dev P@7={m0['P']:.4f} R@7={m0['R']:.4f} F1@7={m0['F1']:.4f} "
            f"(pooled only {m0['F1_pooled']:.4f})")

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t_train0, last_epoch_s, good_steps = time.time(), 0.0, 0
    docs_seen, t_docs = 0, 0.0

    for epoch in range(state["epoch"], C.EPOCHS):
        if (time.time() - T_START) / 3600 + 1.3 * last_epoch_s / 3600 > C.MAX_RUNTIME_H:
            log("time budget would be exceeded by another epoch -> stopping (state is saved, re-run to continue)")
            break
        t_ep = time.time()
        ENC.train()
        rng = np.random.default_rng(C.SEED + 1000 * epoch)
        perm = rng.permutation(len(train_qs))
        run_loss, run_acc, run_n = 0.0, 0.0, 0
        for step in range(steps_per_epoch):
            t_step = time.time()
            qb = [train_qs[i] for i in perm[step * B:(step + 1) * B]]
            cand_ids, cand_docs, seen = [], [], {}

            def add(c):
                if c not in seen:
                    seen[c] = len(cand_ids)
                    cand_ids.append(c)
                    cand_docs.append(tr.cdocs[tr.c_index[c]])

            pos_sets = []
            for q in qb:                                              # NPOS gold positives per query
                g = tr.gold_in_pool[q]
                pick = rng.choice(len(g), min(C.NPOS, len(g)), replace=False)
                pos_sets.append([g[int(t)] for t in pick])
                for c in pos_sets[-1]:
                    add(c)
            if C.HARD_NEGS:
                for q in qb:                                          # hard negatives (BM25, later BM25 + dense)
                    hn = HARD_NEGS_MAP.get(q) or []
                    if hn:
                        for t in rng.choice(len(hn), min(C.HARD_NEGS, len(hn)), replace=False):
                            add(hn[int(t)])
            pos = np.zeros((len(qb), len(cand_ids)), dtype=bool)
            neg = np.ones((len(qb), len(cand_ids)), dtype=bool)       # other gold cases + the query itself are no negatives
            for i, q in enumerate(qb):
                gset = tr.gold[q]
                for j, c in enumerate(cand_ids):
                    if c == q or c in gset:
                        neg[i, j] = False
                for c in pos_sets[i]:
                    pos[i, seen[c]] = True
            pos_t, neg_t = torch.from_numpy(pos).to(DEVICE), torch.from_numpy(neg).to(DEVICE)

            q_docs = [tr.qdocs[tr.q_index[q]] for q in qb]
            Qd, Qz, Qseg = embed_train(q_docs, rng)
            Cd, Cz, Cseg = embed_train(cand_docs, rng)
            S_pool = (Qd @ Cd.T) / C.TEMP
            loss = C.W_POOLED * multipos_loss(S_pool, pos_t, neg_t)
            if C.MAXSIM:
                q2c, c2q = maxsim_matrix(Qz, Qseg, len(q_docs), Cz, Cseg, len(cand_docs))
                loss = loss + C.W_MAXSIM * multipos_loss(0.5 * (q2c + c2q) / C.TEMP, pos_t, neg_t)

            f = lr_scale(state["gs"], total_steps, warm)
            for gr in opt.param_groups:
                gr["lr"] = gr["base_lr"] * f
            if not torch.isfinite(loss):
                log(f"  non-finite loss at step {state['gs']}, batch skipped")
                opt.zero_grad(set_to_none=True)
                state["gs"] += 1
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            gnorm = float(torch.nn.utils.clip_grad_norm_(all_trainable, C.GRAD_CLIP))
            if math.isfinite(gnorm) and gnorm > 0:
                good_steps += 1
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            state["gs"] += 1
            if state["gs"] == 25 and good_steps == 0:
                raise RuntimeError("25 steps without a finite non-zero gradient - the graph is detached or fp16 overflows")

            with torch.no_grad():
                best_neg = S_pool.masked_fill(~neg_t, -1e4).max(1, keepdim=True).values
                acc = float(((S_pool > best_neg) & pos_t).sum() / pos_t.sum().clamp(min=1))
            run_loss += float(loss)
            run_acc += acc
            run_n += 1
            docs_seen += len(q_docs) + len(cand_docs)
            t_docs += time.time() - t_step
            if (step + 1) % 10 == 0 or step + 1 == steps_per_epoch:
                log(f"[ep {epoch + 1}/{C.EPOCHS}] step {step + 1}/{steps_per_epoch} | loss {run_loss / max(run_n, 1):.4f} | "
                    f"pos>negs {run_acc / max(run_n, 1):.3f} | gnorm {gnorm:.2f} | lr {opt.param_groups[0]['lr']:.2e} | "
                    f"{docs_seen / max(t_docs, 1e-9):.1f} docs/s | elapsed {fmt_time(time.time() - T_START)}")

        last_epoch_s = time.time() - t_ep
        state["epoch"] = epoch + 1
        row = {"epoch": epoch + 1, "train_loss": run_loss / max(run_n, 1), "train_acc": run_acc / max(run_n, 1),
               "epoch_seconds": last_epoch_s}
        if (epoch + 1) % C.EVAL_EVERY == 0 or epoch + 1 == C.EPOCHS:
            m, eval_s = evaluate_dev_now()
            row.update(dev_P=m["P"], dev_R=m["R"], dev_F1=m["F1"], dev_F1_pooled=m["F1_pooled"], dev_eval_seconds=eval_s)
            improved = m["F1"] > state["best_f1"] + 1e-6
            if improved:
                state.update(best_f1=m["F1"], best_epoch=epoch + 1, bad=0)
                save_best(epoch + 1, m["F1"])
            else:
                state["bad"] += 1
            log(f"== epoch {epoch + 1}: train loss {row['train_loss']:.4f} | dev P@7={m['P']:.4f} R@7={m['R']:.4f} "
                f"F1@7={m['F1']:.4f} (pooled only {m['F1_pooled']:.4f}) {'(new best)' if improved else ''} | "
                f"best {state['best_f1']:.4f} @ epoch {state['best_epoch']} | {fmt_time(last_epoch_s)}")
        state["history"].append(row)
        if (epoch + 1) in C.REFRESH_NEGS_AT and epoch + 1 < C.EPOCHS and state["bad"] < C.PATIENCE:
            try:
                refresh_hard_negs()
            except Exception as e:                                    # never lose a run over an optional step
                log(f"  hard-negative refresh failed ({e!r}); continuing with the previous negatives")
        state["hard_negs"] = HARD_NEGS_MAP
        atomic_torch_save({"trainable": trainable_state(ENC), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                           "state": state, "tag": C.P7_TAG}, LATEST)
        atomic_json({"tag": C.P7_TAG, "epoch": state["epoch"], "best_f1": state["best_f1"]}, LATEST_META)
        pd.DataFrame(state["history"]).to_csv(LOG_CSV, index=False)
        if state["bad"] >= C.PATIENCE:
            log(f"early stopping: no dev improvement for {C.PATIENCE} evaluations")
            break

    pd.DataFrame(state["history"]).to_csv(LOG_CSV, index=False)
    RESULTS["training"] = {"history": state["history"], "best_dev_f1": state["best_f1"], "best_epoch": state["best_epoch"],
                           "train_seconds": time.time() - t_train0, "docs_per_s": docs_seen / max(t_docs, 1e-9),
                           "peak_gpu_mem_gb": (torch.cuda.max_memory_allocated() / 2 ** 30) if DEVICE.type == "cuda" else None}
    return state

if C.TRAIN:
    TRAIN_STATE = run_training()
else:
    TRAIN_STATE = None
    log("TRAIN=False -> skipping training")

# continue with the best checkpoint (highest dev F1@7 of the model score)
cands = [BEST] if os.path.exists(BEST) else []
if not cands:
    for d in walk_find("p7_best_meta.json", [INPUT_ROOT]):
        meta = read_json(os.path.join(d, "p7_best_meta.json"))
        if meta and meta.get("tag") == C.P7_TAG and os.path.exists(os.path.join(d, "p7_best.pt")):
            cands.append(os.path.join(d, "p7_best.pt"))
if cands:
    bl = torch.load(cands[0], map_location="cpu", weights_only=False)
    load_trainable_state(ENC, bl["trainable"])
    log(f"loaded best checkpoint {cands[0]} (epoch {bl.get('epoch')}, dev F1@7 {bl.get('dev_f1', float('nan')):.4f})")
else:
    log("no trained checkpoint found - continuing with the model as it is")

### 9. Dev evaluation, score fusion, failure analysis

In [ ]:

# Validation and score fusion

# q2c, c2q = chunk-level MaxSim (only if MAXSIM)
# bm25, refs = lexical BM25 and statute/article/Act overlap

dev = SP["dev"]
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
raw_dev = encode_split_raw(ENC, dev, need=(C.TOKEN_POOL,))
comps_dev, Qe_dev, Ce_dev = model_components(ENC, raw_dev)
RESULTS["efficiency_adapted"] = {
    "dev_docs": raw_dev["n_docs"], "dev_chunks": raw_dev["n_chunks"], "seconds": raw_dev["seconds"],
    "docs_per_s": raw_dev["n_docs"] / max(raw_dev["seconds"], 1e-9),
    "peak_gpu_mem_gb": (torch.cuda.max_memory_allocated() / 2 ** 30) if DEVICE.type == "cuda" else None}
log(f"[adapted] encoded dev in {fmt_time(raw_dev['seconds'])} ({RESULTS['efficiency_adapted']['docs_per_s']:.1f} docs/s)")
del raw_dev
gc.collect()

dq, dc = [dev.qtext[q] for q in dev.qids], [dev.ctext[c] for c in dev.cids]

def lexical_components(q_texts, c_texts):
    out = {}
    if C.USE_BM25:
        out["bm25"] = bm25_scores(q_texts, c_texts, C.BM25_NGRAM, C.BM25_K1, C.BM25_B)
    if C.USE_REFS:
        out["refs"] = refs_scores(q_texts, c_texts)
    return out

def on_weights(w):
    return {k: v for k, v in w.items() if v}

FUSION_OK = True
try:
    lex_dev = lexical_components(dq, dc)
    Z_dev = {k: zrows(v) for k, v in {**comps_dev, **lex_dev}.items()}
    singles = []
    for k, Z in Z_dev.items():
        mm = micro_prf(rank_topk(Z, dev.qids, dev.cids, C.K), dev.gold, dev.qids, C.K)
        singles.append({"component": k, "P@7": round(mm["P"], 4), "R@7": round(mm["R"], 4), "F1@7": round(mm["F1"], 4)})
    S_model_dev = model_score(comps_dev)
    mm = micro_prf(rank_topk(S_model_dev, dev.qids, dev.cids, C.K), dev.gold, dev.qids, C.K)
    singles.append({"component": "model score (selection metric)", "P@7": round(mm["P"], 4), "R@7": round(mm["R"], 4), "F1@7": round(mm["F1"], 4)})
    log("[adapted] dev, every score component on its own:")
    log(pd.DataFrame(singles).to_string(index=False))
    RESULTS["components_dev"] = singles

    W, f_fused, f_pooled = tune_fusion(Z_dev, "pooled", dev, C.K, min_gain=C.MIN_GAIN)
    S_fused_dev = fuse(Z_dev, W)
    log(f"[fusion] weights {on_weights(W)} -> dev F1@7 {f_fused:.4f}  (pooled alone {f_pooled:.4f})")
    PRF_M, PRF_G = None, 0.0
    f_prf = f_fused
    if C.USE_PRF:
        PRF_M, PRF_G, f_prf, _ = tune_prf(S_fused_dev, Ce_dev, dev, C.K, min_gain=C.MIN_GAIN)
        prf_txt = "off (no gain >= MIN_GAIN)" if PRF_M is None else "top-%d centroid, gamma %s" % (PRF_M, PRF_G)
        log(f"[PRF] {prf_txt} -> dev F1@7 {f_prf:.4f}")
    S_final_dev = S_fused_dev if PRF_M is None else S_fused_dev + PRF_G * prf_component(S_fused_dev, Ce_dev, dev.qids, dev.cids, PRF_M)
    RESULTS["fusion"] = {"weights": W, "prf_m": PRF_M, "prf_gamma": PRF_G, "dev_f1_pooled": f_pooled,
                         "dev_f1_fused": f_fused, "dev_f1_fused_prf": f_prf}
except Exception as e:                                              # fall back to the model alone, never lose the run
    FUSION_OK = False
    log(f"!! fusion failed ({e!r}); falling back to the model score alone")
    W, PRF_M, PRF_G = {"pooled": 1.0}, None, 0.0
    S_final_dev = model_score(comps_dev)
    Z_dev = {k: zrows(v) for k, v in comps_dev.items()}

preds_dev = rank_topk(S_final_dev, dev.qids, dev.cids, C.K)
M_DEV = micro_prf(preds_dev, dev.gold, dev.qids, C.K)
S_model_dev = model_score(comps_dev)
M_DEV_MODEL = micro_prf(rank_topk(S_model_dev, dev.qids, dev.cids, C.K), dev.gold, dev.qids, C.K)
log(f"\n>>> DEV  model alone : P@7={M_DEV_MODEL['P']:.4f}  R@7={M_DEV_MODEL['R']:.4f}  F1@7={M_DEV_MODEL['F1']:.4f}")
log(f">>> DEV  final system: P@7={M_DEV['P']:.4f}  R@7={M_DEV['R']:.4f}  F1@7={M_DEV['F1']:.4f}   "
    f"(fusion={'on' if FUSION_OK else 'FAILED'}, weights {on_weights(W)}, PRF={PRF_M})")
RESULTS["dev_final"] = {**M_DEV, "weights": W, "prf_m": PRF_M, "prf_gamma": PRF_G}
RESULTS["dev_model_only"] = M_DEV_MODEL

# per-query failure analysis
PQ = per_query_table(S_final_dev, dev, preds_dev, C.K)
PQ["f1"] = 2 * PQ["hits"] / (C.K + PQ["n_gold"])
PQ.to_csv(os.path.join(OUT_DIR, "dev_per_query.csv"), index=False)
n_zero = int((PQ["hits"] == 0).sum())
_med = PQ.loc[PQ.first_hit_rank > 0, "first_hit_rank"].median()
log(f"queries with 0 hits in the top-7: {n_zero}/{len(PQ)} ({100 * n_zero / len(PQ):.1f}%) | "
    f"median rank of the first gold case: {int(_med) if _med == _med else 'n/a'}")
try:
    PQ["len_bucket"] = pd.qcut(PQ["q_tokens"], q=min(5, PQ["q_tokens"].nunique()), duplicates="drop")
except ValueError:
    PQ["len_bucket"] = "all"
PQ["gold_bucket"] = pd.cut(PQ["n_gold"], bins=[0, 1, 3, 6, 10 ** 6], labels=["1", "2-3", "4-6", "7+"])
for col, title in (("len_bucket", "query length (tokens)"), ("gold_bucket", "number of gold cases")):
    log(f"\nby {title}:")
    log(PQ.groupby(col, observed=True).agg(queries=("qid", "count"), recall=("recall", "mean"), precision=("precision", "mean"),
                                          zero_hit=("hits", lambda s: (s == 0).mean())).round(3).to_string())

np.savez_compressed(os.path.join(OUT_DIR, "scores_dev.npz"), qids=np.array(dev.qids), cids=np.array(dev.cids),
                    final=S_final_dev.astype(np.float16), model=S_model_dev.astype(np.float16),
                    **{f"comp_{k}": v.astype(np.float16) for k, v in Z_dev.items()})
np.savez_compressed(os.path.join(OUT_DIR, "dev_embeddings.npz"), Q=Qe_dev, C=Ce_dev, qids=np.array(dev.qids), cids=np.array(dev.cids))

### 10. Test predictions

In [ ]:

# Test predictions

# Return seven unique candidates per test query.

def write_submission(pred, qids, pool, folder):
    os.makedirs(folder, exist_ok=True)
    pool = set(pool)
    for q in qids:
        p = pred[q]
        assert len(p) == C.K and len(set(p)) == C.K, f"query {q}: need {C.K} unique ids"
        assert all(c in pool for c in p), f"id outside the test pool for query {q}"
        assert q not in p, f"query {q} retrieved itself"
    path = os.path.join(folder, "prediction.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump({q: pred[q] for q in qids}, f, indent=2)
    chk = json.load(open(path, encoding="utf-8"))
    assert list(chk.keys()) == list(qids) and all(len(v) == C.K and len(set(v)) == C.K for v in chk.values())
    with zipfile.ZipFile(os.path.join(folder, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")
    return path, len(chk)

if C.RUN_PREDICT:
    te = SP["test"]
    raw_te = encode_split_raw(ENC, te, need=(C.TOKEN_POOL,))
    comps_te, Qe_te, Ce_te = model_components(ENC, raw_te)
    S_model_te = model_score(comps_te)

    # Save model-only predictions first.
    p0, n0 = write_submission(rank_topk(S_model_te, te.qids, te.cids, C.K), te.qids, te.cids, os.path.join(OUT_DIR, "model_only"))
    log(f"wrote {p0}  ({n0} queries)")

    # Save the final fused predictions.
    S_final_te, Z_te = S_model_te, {k: zrows(v) for k, v in comps_te.items()}
    if FUSION_OK:
        try:
            lex_te = lexical_components([te.qtext[q] for q in te.qids], [te.ctext[c] for c in te.cids])
            Z_te = {k: zrows(v) for k, v in {**comps_te, **lex_te}.items()}
            S_final_te = fuse(Z_te, W)
            if PRF_M is not None:
                S_final_te = S_final_te + PRF_G * prf_component(S_final_te, Ce_te, te.qids, te.cids, PRF_M)
        except Exception as e:
            log(f"!! test-time fusion failed ({e!r}); the final file contains the model-only ranking")
            S_final_te = S_model_te
    pred_te = rank_topk(S_final_te, te.qids, te.cids, C.K)
    path, n = write_submission(pred_te, te.qids, te.cids, OUT_DIR)
    shutil.copy(os.path.join(OUT_DIR, "submission.zip"), os.path.join(OUT_DIR, "submission_final.zip"))
    np.savez_compressed(os.path.join(OUT_DIR, "scores_test.npz"), qids=np.array(te.qids), cids=np.array(te.cids),
                        final=S_final_te.astype(np.float16), model=S_model_te.astype(np.float16),
                        **{f"comp_{k}": v.astype(np.float16) for k, v in Z_te.items()})
    np.savez_compressed(os.path.join(OUT_DIR, "test_embeddings.npz"), Q=Qe_te, C=Ce_te, qids=np.array(te.qids), cids=np.array(te.cids))
    log(f"wrote {path}  ({n} test queries x {C.K} ids) + submission.zip")
    if C.MAX_QUERIES or C.MAX_POOL:
        log("WARNING: this was a smoke run (MAX_QUERIES / MAX_POOL set) - the file covers only part of the test set. DO NOT SUBMIT IT.")
    log("example:", list(json.load(open(path)).items())[0])
    RESULTS["test"] = {"queries": n, "pool": len(te.cids), "encode_seconds": raw_te["seconds"]}
    del raw_te
    gc.collect()
else:
    log("RUN_PREDICT=False -> no prediction file written")

### 11. Summary for the report

In [ ]:

# Results summary

def _best_zero_shot():
    g = RESULTS.get("zero_shot_grid")
    return max(g, key=lambda r: r["F1@7"]) if g else None

zs = _best_zero_shot()
rows = []
for k_, m_ in RESULTS.get("lexical_dev", {}).items():
    rows.append({"system": f"{k_} (classical baseline)", "P@7": m_["P"], "R@7": m_["R"], "F1@7": m_["F1"], "trainable %": 0.0})
if zs:
    rows.append({"system": f"Phase-1 zero-shot ({zs['token_pool']}/{zs['doc_pool']}/centre={zs['center']})",
                 "P@7": zs["P@7"], "R@7": zs["R@7"], "F1@7": zs["F1@7"], "trainable %": 0.0})
pct = RESULTS["params"]["trainable_percent"]
rows.append({"system": f"{C.EXP_NAME}: model alone", "P@7": M_DEV_MODEL["P"], "R@7": M_DEV_MODEL["R"],
             "F1@7": M_DEV_MODEL["F1"], "trainable %": pct})
rows.append({"system": f"{C.EXP_NAME}: + fusion/PRF (final)", "P@7": M_DEV["P"], "R@7": M_DEV["R"],
             "F1@7": M_DEV["F1"], "trainable %": pct})
SUMMARY = pd.DataFrame(rows)
log(SUMMARY.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

ea, P_ = RESULTS.get("efficiency_adapted", {}), RESULTS["params"]
log(f"\nTECHNIQUE: {P_['adaptation']}")
log(f"PARAMETERS: {P_['total_params'] / 1e6:.2f}M total | {P_['trainable_params'] / 1e6:.3f}M trainable = {P_['trainable_percent']:.2f}% | "
    f"inference {ea.get('docs_per_s', float('nan')):.1f} docs/s | peak GPU memory (inference) {ea.get('peak_gpu_mem_gb')} GB | "
    f"training peak {RESULTS.get('training', {}).get('peak_gpu_mem_gb')} GB")
RESULTS["summary"] = rows
RESULTS["total_runtime_s"] = time.time() - T_START

def _jsonable(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (set, tuple)):
        return list(o)
    return str(o)

with open(os.path.join(OUT_DIR, f"results_{C.EXP_NAME}.json"), "w", encoding="utf-8") as f:
    json.dump(RESULTS, f, indent=1, default=_jsonable)
if C.VARIANT == "lormaplus":                                         # the adapters are small enough to keep next to the outputs
    for fn in ("p7_best.pt",):
        src = os.path.join(CKPT_DIR, fn)
        if os.path.exists(src):
            shutil.copy(src, os.path.join(OUT_DIR, fn))
if os.path.exists(os.path.join(CKPT_DIR, "p7_train_log.csv")):
    shutil.copy(os.path.join(CKPT_DIR, "p7_train_log.csv"), os.path.join(OUT_DIR, "p7_train_log.csv"))
log(f"\nartifacts in {OUT_DIR}: " + ", ".join(sorted(os.listdir(OUT_DIR))))
log(f"total runtime {fmt_time(time.time() - T_START)}")